In [ ]:
# ========== 1. IMPORT THƯ VIỆN ========== #
import os
import textwrap
import numpy as np
import tensorflow as tf   # <- tf phải được tạo trước
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc


# ============================================================================
# PAPER-READY FIGURE AND REPORT HELPERS
# ============================================================================
# These helpers keep labels, titles, and saved reports consistent across all
# runs. They use readable class names for figures while preserving the raw
# folder names for data loading and indexing.
EXPERIMENT_DISPLAY_NAME = "NASNetMobile head-only fine-tuning"

CLASS_NAME_MAP = {
    "Fully_Peeled_Garlic": "Fully peeled garlic",
    "Partially_Peeled_Garlic": "Partially peeled garlic",
    "Spoiled_Garlic": "Spoiled garlic",
}


def format_class_label(label):
    """Return a reader-facing class label for paper figures and reports."""
    raw = str(label)
    if raw in CLASS_NAME_MAP:
        return CLASS_NAME_MAP[raw]
    text = raw.replace("_", " ").strip()
    if not text:
        return raw
    return text[:1].upper() + text[1:].lower()


def plain_class_names(labels):
    """Return unwrapped display labels for tables, legends, and text reports."""
    return [format_class_label(label) for label in labels]


def display_class_names(labels, width=18):
    """Return wrapped display labels so class ticks fit compact figures."""
    return ["\n".join(textwrap.wrap(format_class_label(label), width=max(1, width)))
            for label in labels]


def experiment_display_name():
    """Return the model/strategy label used in scientific figure titles."""
    for key in ("STRATEGY_LABEL", "MODEL_LABEL", "EXPERIMENT_DISPLAY_NAME"):
        value = globals().get(key)
        if isinstance(value, str) and value.strip():
            return value.strip()
    return "Garlic classification model"


def format_confusion_matrix_axes(ax):
    """Apply consistent axis labels and tick layout to confusion matrices."""
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.tick_params(axis="x", labelrotation=35)
    for tick in ax.get_xticklabels():
        tick.set_ha("right")
    for tick in ax.get_yticklabels():
        tick.set_rotation(0)
        tick.set_va("center")


In [2]:
# ========== GPU CONFIG (KAGGLE) ========== #
print("TensorFlow version:", tf.__version__)
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU is ready ✅")
    except RuntimeError as e:
        print(e)

# confirm mixed precision
print("Compute dtype:", tf.keras.mixed_precision.global_policy().compute_dtype)
print("Variable dtype:", tf.keras.mixed_precision.global_policy().variable_dtype)

TensorFlow version: 2.19.0
Num GPUs Available: 2
GPU is ready ✅
Compute dtype: float32
Variable dtype: float32


In [ ]:
## NASNetMobile 21-06 - MULTI-RUN EXPERIMENT
# ========== 1. IMPORT THƯ VIỆN ========== #
import os
import numpy as np
import tensorflow as tf
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import NASNetMobile
from tensorflow.keras.applications.nasnet import preprocess_input as nasnet_preprocess
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from tensorflow.keras.losses import CategoricalCrossentropy
from sklearn.utils import class_weight
from sklearn.metrics import classification_report, confusion_matrix, top_k_accuracy_score
from tensorflow.keras.regularizers import l2

# ========== 2. THIẾT LẬP MIXED PRECISION ========== #
tf.keras.mixed_precision.set_global_policy("mixed_float16")

# ========== 3. DATA PATH & CONFIGURATION ========== #
DATA_DIR = "/kaggle/input/datasets/giaphuc/dataset-0803/dataset_split_0803"
BASE_RESULT_DIR = "/kaggle/working/report_NASNetMobile_MultiRun"
os.makedirs(BASE_RESULT_DIR, exist_ok=True)

# RANDOM SEEDS FOR EXPERIMENTS
RANDOM_SEEDS = [42, 123, 456]
print(f"🔬 Running {len(RANDOM_SEEDS)} experiments with seeds: {RANDOM_SEEDS}")

# Store results from all runs
all_runs_results = []

# ========== 4. FUNCTION DEFINITIONS ========== #
def create_generators(data_dir, input_size, batch_size=32, seed=None):
    train_datagen = ImageDataGenerator(
        preprocessing_function=nasnet_preprocess,
        rotation_range=30,
        width_shift_range=0.2,
        height_shift_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
        vertical_flip=True,
        brightness_range=[0.7, 1.3],
        fill_mode='nearest'
    )
    val_datagen = ImageDataGenerator(preprocessing_function=nasnet_preprocess)

    train_gen = train_datagen.flow_from_directory(
        os.path.join(data_dir, 'train'),
        target_size=input_size,
        batch_size=batch_size,
        class_mode='categorical',
        seed=seed
    )
    val_gen = val_datagen.flow_from_directory(
        os.path.join(data_dir, 'val'),
        target_size=input_size,
        batch_size=batch_size,
        class_mode='categorical',
        seed=seed
    )
    test_gen = val_datagen.flow_from_directory(
        os.path.join(data_dir, 'test'),
        target_size=input_size,
        batch_size=batch_size,
        class_mode='categorical',
        shuffle=False
    )
    return train_gen, val_gen, test_gen

# ========== 5. MAIN EXPERIMENT LOOP ========== #
input_shape = (224, 224, 3)  # NASNetMobile default input size
batch_size = 32
epochs = 50

for run_idx, seed in enumerate(RANDOM_SEEDS):
    print("\n" + "="*80)
    print(f"🚀 STARTING RUN {run_idx + 1}/{len(RANDOM_SEEDS)} - SEED: {seed}")
    print("="*80 + "\n")
    
    # Set random seed for reproducibility
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    
    # Create result directory for this run
    RESULT_DIR = os.path.join(BASE_RESULT_DIR, f"run_{run_idx+1}_seed_{seed}")
    os.makedirs(RESULT_DIR, exist_ok=True)
    
    # Load data with seed
    train_generator, val_generator, test_generator = create_generators(
        DATA_DIR, input_shape[:2], batch_size, seed=seed
    )
    
    class_weights = class_weight.compute_class_weight(
        class_weight='balanced',
        classes=np.unique(train_generator.classes),
        y=train_generator.classes
    )
    class_weights = dict(enumerate(class_weights))
    
    # Build model
    base_model = NASNetMobile(weights='imagenet', include_top=False, input_shape=input_shape)
    base_model.trainable = False
    
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dense(128, activation='relu', kernel_regularizer=l2(1e-5))(x)
    x = Dropout(0.5)(x)
    outputs = Dense(len(train_generator.class_indices), activation='softmax', dtype='float32')(x)
    
    model = Model(inputs=base_model.input, outputs=outputs)
    
    # Compile
    steps_per_epoch = train_generator.samples // batch_size
    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=1e-5,
        decay_steps=steps_per_epoch * 5,
        decay_rate=0.9,
        staircase=True
    )
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)
    
    model.compile(
        optimizer=optimizer,
        loss=CategoricalCrossentropy(label_smoothing=0.15),
        metrics=['accuracy']
    )
    
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
        CSVLogger(os.path.join(RESULT_DIR, 'training_log.csv'), append=False),
        ModelCheckpoint(os.path.join(RESULT_DIR, 'nasnetmobile_best.keras'),
                        save_best_only=True, monitor='val_loss', verbose=1)
    ]
    
    # Train
    history = model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=epochs,
        class_weight=class_weights,
        callbacks=callbacks
    )
    
    # Save learning curves
    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1)
    plt.plot(history.history['accuracy'], label='Train Acc')
    plt.plot(history.history['val_accuracy'], label='Val Acc')
    plt.title(f'Accuracy trajectories (run {run_idx + 1}, seed {seed})')
    plt.legend()
    plt.grid()
    plt.subplot(1,2,2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.title(f'Loss trajectories (run {run_idx + 1}, seed {seed})')
    plt.legend()
    plt.grid()
    plt.savefig(os.path.join(RESULT_DIR, "learning_curve.png"), dpi=300)
    plt.close()
    
    # Load best model and evaluate
    model = load_model(os.path.join(RESULT_DIR, 'nasnetmobile_best.keras'))
    test_generator.reset()
    
    # Predictions
    pred_probs = model.predict(test_generator, verbose=1)
    y_pred = np.argmax(pred_probs, axis=1)
    y_true = test_generator.classes
    class_names = list(test_generator.class_indices.keys())
    
    # Classification report
    report = classification_report(y_true, y_pred, target_names=class_names, 
                                   output_dict=True, digits=4)
    
    # Save text report
    report_text = classification_report(y_true, y_pred, target_names=class_names, digits=4)
    with open(os.path.join(RESULT_DIR, "classification_report.txt"), "w") as f:
        f.write(report_text)
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(7,6))
    sns.heatmap(cm, annot=True, fmt='d',
                xticklabels=display_class_names(class_names),
                yticklabels=display_class_names(class_names),
                cmap='Blues')
    plt.xlabel("Predicted class")
    plt.ylabel("True class")
    format_confusion_matrix_axes(plt.gca())
    plt.title(f"{experiment_display_name()}: test-set confusion matrix (run {run_idx + 1}, seed {seed})", fontweight="bold")
    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.savefig(os.path.join(RESULT_DIR, "confusion_matrix.png"), dpi=300)
    plt.close()
    
    # Collect metrics for this run
    test_acc = np.mean(y_pred == y_true)
    
    # Overall metrics
    overall_precision = report['weighted avg']['precision']
    overall_recall = report['weighted avg']['recall']
    overall_f1 = report['weighted avg']['f1-score']
    
    # Per-class metrics
    per_class_metrics = {}
    for class_name in class_names:
        per_class_metrics[class_name] = {
            'precision': report[class_name]['precision'],
            'recall': report[class_name]['recall'],
            'f1-score': report[class_name]['f1-score']
        }
    
    run_results = {
        'run': run_idx + 1,
        'seed': seed,
        'accuracy': test_acc,
        'precision': overall_precision,
        'recall': overall_recall,
        'f1_score': overall_f1,
        'per_class_metrics': per_class_metrics,
        'class_names': class_names,
        'result_dir': RESULT_DIR,
        'history': history.history,
        'y_true': y_true,
        'y_pred': y_pred,
        'pred_probs': pred_probs,
        'test_filenames': test_generator.filenames,
        'n_train': train_generator.samples,
        'n_val': val_generator.samples,
        'n_test': test_generator.samples
    }
    
    all_runs_results.append(run_results)
    
    print(f"\n✅ RUN {run_idx + 1} COMPLETED")
    print(f"   Accuracy: {test_acc:.4f}")
    print(f"   Precision: {overall_precision:.4f}")
    print(f"   Recall: {overall_recall:.4f}")
    print(f"   F1-Score: {overall_f1:.4f}")
    
    # Clear session to free memory
    tf.keras.backend.clear_session()

print("\n" + "="*80)
print("🎉 ALL EXPERIMENTS COMPLETED!")
print("="*80)

In [ ]:
# ========== STANDARDIZED PER-RUN REPORTS AND FIGURES ========== #
# Run this cell after the training loop. It exports the same paper-ready report
# package for every completed seed, not only the manually selected run.
def _metric_rows_for_run(run_data):
    """Build the scalar metric table saved for each run."""
    metric_keys = [
        ("Accuracy", "accuracy"),
        ("Weighted precision", "precision"),
        ("Weighted recall", "recall"),
        ("Weighted F1-score", "f1_score"),
        ("Macro AUC", "auc_macro"),
        ("Weighted AUC", "auc_weighted"),
        ("Macro OvR AUC", "auc_ovr_macro"),
        ("Weighted OvR AUC", "auc_ovr_weighted"),
        ("Balanced accuracy", "bal_acc"),
        ("Cohen's kappa", "kappa"),
        ("Matthews correlation coefficient", "mcc"),
    ]
    rows = []
    for label, key in metric_keys:
        if key in run_data and run_data[key] is not None:
            rows.append({"Metric": label, "Value": float(run_data[key])})
    return rows


def _save_standardized_run_artifacts(run_data):
    """Save standardized test-set reports and diagnostic figures for one run."""
    result_dir = run_data.get("result_dir")
    if not result_dir:
        print("Skipped a run because result_dir is missing.")
        return
    os.makedirs(result_dir, exist_ok=True)

    class_names_local = run_data.get("class_names") or list(run_data.get("per_class_metrics", {}).keys())
    y_true_local = run_data.get("y_true")
    y_pred_local = run_data.get("y_pred")
    pred_probs_local = run_data.get("pred_probs")

    if y_true_local is None or y_pred_local is None or not class_names_local:
        print(f"Skipped standardized artifacts for {result_dir}: missing y_true/y_pred/class_names.")
        return

    y_true_local = np.asarray(y_true_local)
    y_pred_local = np.asarray(y_pred_local)
    run_no = run_data.get("run", "?")
    seed = run_data.get("seed", "?")
    model_label = experiment_display_name()

    # Scalar metric table.
    metric_rows = _metric_rows_for_run(run_data)
    if metric_rows:
        pd.DataFrame(metric_rows).to_csv(
            os.path.join(result_dir, "run_metrics_summary.csv"), index=False)

    # Per-class metric table with readable class labels.
    per_class_metrics = run_data.get("per_class_metrics")
    if not per_class_metrics:
        per_class_metrics = classification_report(
            y_true_local, y_pred_local,
            target_names=class_names_local, output_dict=True, digits=4)
    per_class_rows = []
    for raw_name in class_names_local:
        metrics = per_class_metrics.get(raw_name, {})
        per_class_rows.append({
            "Class": format_class_label(raw_name),
            "Source label": raw_name,
            "Precision": metrics.get("precision", np.nan),
            "Recall": metrics.get("recall", np.nan),
            "F1-score": metrics.get("f1-score", np.nan),
            "Support": metrics.get("support", int(np.sum(y_true_local == class_names_local.index(raw_name))) if raw_name in class_names_local else np.nan),
        })
    pd.DataFrame(per_class_rows).to_csv(
        os.path.join(result_dir, "per_class_metrics_readable.csv"), index=False)

    with open(os.path.join(result_dir, "classification_report_readable.txt"), "w", encoding="utf-8") as f:
        f.write(classification_report(
            y_true_local, y_pred_local,
            target_names=plain_class_names(class_names_local), digits=4))

    # Count and row-normalized confusion matrices.
    cm = confusion_matrix(y_true_local, y_pred_local)
    cm_norm = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.8))
    labels = display_class_names(class_names_local, width=16)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels, ax=axes[0], cbar=True)
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
                xticklabels=labels, yticklabels=labels, ax=axes[1], cbar=True)
    axes[0].set_title("Confusion matrix counts", fontweight="bold")
    axes[1].set_title("Row-normalized recall matrix", fontweight="bold")
    for axis in axes:
        format_confusion_matrix_axes(axis)
    fig.suptitle(f"{model_label}: test-set confusion matrix analysis for run {run_no} (seed {seed})",
                 fontsize=12, fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.90])
    fig.savefig(os.path.join(result_dir, "confusion_matrix_standardized.png"),
                dpi=300, bbox_inches="tight")
    plt.close(fig)

    # One-vs-rest ROC analysis when probabilities are available.
    if pred_probs_local is not None:
        pred_probs_local = np.asarray(pred_probs_local)
        if pred_probs_local.ndim == 2 and pred_probs_local.shape[1] == len(class_names_local):
            y_bin = label_binarize(y_true_local, classes=range(len(class_names_local)))
            fig, ax = plt.subplots(figsize=(7.5, 6))
            auc_rows = []
            for ci, class_name in enumerate(class_names_local):
                fpr, tpr, _ = roc_curve(y_bin[:, ci], pred_probs_local[:, ci])
                auc_value = auc(fpr, tpr)
                auc_rows.append({"Class": format_class_label(class_name), "AUC": auc_value})
                ax.plot(fpr, tpr, lw=2, label=f"{format_class_label(class_name)} (AUC={auc_value:.3f})")
            ax.plot([0, 1], [0, 1], "--", color="0.55", lw=1)
            ax.set_xlim([0, 1])
            ax.set_ylim([0, 1.01])
            ax.set_xlabel("False positive rate")
            ax.set_ylabel("True positive rate")
            ax.set_title(f"One-vs-rest ROC analysis (run {run_no}, seed {seed})", fontweight="bold")
            ax.legend(loc="lower right", fontsize=8)
            ax.grid(alpha=0.3)
            fig.tight_layout()
            fig.savefig(os.path.join(result_dir, "roc_curves_ovr_standardized.png"),
                        dpi=300, bbox_inches="tight")
            plt.close(fig)
            pd.DataFrame(auc_rows).to_csv(os.path.join(result_dir, "auc_scores_readable.csv"), index=False)

    # Learning curves from the stored Keras history.
    hist = run_data.get("history") or {}
    if "loss" in hist:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
        axes[0].plot(hist.get("loss", []), label="Training loss", lw=2)
        if "val_loss" in hist:
            axes[0].plot(hist.get("val_loss", []), label="Validation loss", lw=2)
        axes[0].set_title("Loss trajectories", fontweight="bold")
        axes[0].set_xlabel("Epoch")
        axes[0].set_ylabel("Loss")
        axes[0].legend()
        axes[0].grid(alpha=0.3)

        axes[1].plot(hist.get("accuracy", []), label="Training accuracy", lw=2)
        if "val_accuracy" in hist:
            axes[1].plot(hist.get("val_accuracy", []), label="Validation accuracy", lw=2)
        axes[1].set_title("Accuracy trajectories", fontweight="bold")
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("Accuracy")
        axes[1].legend()
        axes[1].grid(alpha=0.3)
        fig.suptitle(f"{model_label}: learning curves for run {run_no} (seed {seed})",
                     fontsize=12, fontweight="bold")
        fig.tight_layout(rect=[0, 0, 1, 0.90])
        fig.savefig(os.path.join(result_dir, "learning_curves_standardized.png"),
                    dpi=300, bbox_inches="tight")
        plt.close(fig)

    summary_lines = [
        "=" * 80,
        "STANDARDIZED SINGLE-RUN EVALUATION REPORT",
        "=" * 80,
        f"Model/strategy: {model_label}",
        f"Run: {run_no}",
        f"Seed: {seed}",
        f"Result directory: {result_dir}",
        f"Classes: {', '.join(plain_class_names(class_names_local))}",
        "",
        "TEST-SET METRICS",
        "-" * 80,
    ]
    for row in metric_rows:
        summary_lines.append(f"{row['Metric']}: {row['Value']:.4f}")
    summary_lines.extend([
        "",
        "GENERATED ARTIFACTS",
        "-" * 80,
        "run_metrics_summary.csv",
        "per_class_metrics_readable.csv",
        "classification_report_readable.txt",
        "confusion_matrix_standardized.png",
        "roc_curves_ovr_standardized.png (when probability outputs are available)",
        "learning_curves_standardized.png (when history is available)",
    ])
    with open(os.path.join(result_dir, "RUN_EVALUATION_REPORT.txt"), "w", encoding="utf-8") as f:
        f.write("\n".join(summary_lines))


for _run_data in all_runs_results:
    _save_standardized_run_artifacts(_run_data)

print(f"Standardized per-run artifacts exported for {len(all_runs_results)} completed run(s).")


In [ ]:
# ========== PAPER-READY ARTIFACT EXPORT SUITE ========== #
# Canonical outputs for thesis/paper writing are saved under:
#   BASE_RESULT_DIR/paper_artifacts/
# Legacy files are left untouched for backward compatibility.

PAPER_ARTIFACT_DIR = os.path.join(BASE_RESULT_DIR, "paper_artifacts")
os.makedirs(PAPER_ARTIFACT_DIR, exist_ok=True)


def _apply_paper_style():
    """Apply a consistent visual style for publication-ready figures."""
    sns.set_theme(style="whitegrid", context="paper", font_scale=1.0)
    plt.rcParams.update({
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "axes.titlesize": 12,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 8,
        "axes.titleweight": "bold",
        "axes.grid": True,
        "grid.alpha": 0.25,
        "font.family": "DejaVu Sans",
    })


def _paper_metric_value(run_data, *keys):
    """Return the first available metric value from a run dictionary."""
    for key in keys:
        value = run_data.get(key)
        if value is not None:
            try:
                return float(value)
            except (TypeError, ValueError):
                return value
    return np.nan


def _paper_run_label(run_data):
    """Return a compact run label used in tables and figure legends."""
    return f"Run {run_data.get('run', '?')} (seed {run_data.get('seed', '?')})"


def _paper_run_dir(run_data):
    """Return the canonical artifact directory for one run."""
    run_no = int(run_data.get("run", 0) or 0)
    seed = run_data.get("seed", "unknown")
    out_dir = os.path.join(PAPER_ARTIFACT_DIR, f"run_{run_no:02d}_seed_{seed}")
    os.makedirs(out_dir, exist_ok=True)
    return out_dir


def _paper_safe_array(value):
    """Convert stored numpy/list values to an array when available."""
    if value is None:
        return None
    return np.asarray(value)


def _paper_metrics_dataframe(runs):
    """Build the canonical run-level test metric table."""
    rows = []
    for run_data in runs:
        rows.append({
            "Run": run_data.get("run"),
            "Seed": run_data.get("seed"),
            "Accuracy": _paper_metric_value(run_data, "accuracy"),
            "Weighted precision": _paper_metric_value(run_data, "precision"),
            "Weighted recall": _paper_metric_value(run_data, "recall"),
            "Weighted F1-score": _paper_metric_value(run_data, "f1_score"),
            "Macro AUC": _paper_metric_value(run_data, "auc_macro", "auc_ovr_macro"),
            "Weighted AUC": _paper_metric_value(run_data, "auc_weighted", "auc_ovr_weighted"),
            "Balanced accuracy": _paper_metric_value(run_data, "bal_acc"),
            "Cohen's kappa": _paper_metric_value(run_data, "kappa"),
            "Matthews correlation coefficient": _paper_metric_value(run_data, "mcc"),
        })
    df = pd.DataFrame(rows)
    metric_cols = [c for c in df.columns if c not in ["Run", "Seed"]]
    for col in metric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def _paper_metrics_summary(metrics_df):
    """Summarize run-level metrics as mean, SD, and number of valid runs."""
    rows = []
    for col in [c for c in metrics_df.columns if c not in ["Run", "Seed"]]:
        values = pd.to_numeric(metrics_df[col], errors="coerce").dropna()
        if len(values) == 0:
            continue
        rows.append({
            "Metric": col,
            "Mean": float(values.mean()),
            "SD": float(values.std(ddof=0)),
            "N": int(len(values)),
        })
    return pd.DataFrame(rows)


def _paper_per_class_dataframe(runs):
    """Build the canonical per-class metric table across all runs."""
    rows = []
    for run_data in runs:
        class_names_local = run_data.get("class_names") or list(run_data.get("per_class_metrics", {}).keys())
        metrics = run_data.get("per_class_metrics", {})
        for class_name in class_names_local:
            class_metrics = metrics.get(class_name, {})
            rows.append({
                "Run": run_data.get("run"),
                "Seed": run_data.get("seed"),
                "Class": format_class_label(class_name),
                "Source label": class_name,
                "Precision": class_metrics.get("precision", np.nan),
                "Recall": class_metrics.get("recall", np.nan),
                "F1-score": class_metrics.get("f1-score", np.nan),
                "Support": class_metrics.get("support", np.nan),
            })
    df = pd.DataFrame(rows)
    for col in ["Precision", "Recall", "F1-score", "Support"]:
        if col in df:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def _paper_per_class_summary(per_class_df):
    """Summarize per-class precision, recall, and F1-score across runs."""
    rows = []
    if per_class_df.empty:
        return pd.DataFrame(rows)
    for class_name, group in per_class_df.groupby("Class", sort=False):
        for metric in ["Precision", "Recall", "F1-score"]:
            values = pd.to_numeric(group[metric], errors="coerce").dropna()
            if len(values) == 0:
                continue
            rows.append({
                "Class": class_name,
                "Metric": metric,
                "Mean": float(values.mean()),
                "SD": float(values.std(ddof=0)),
                "N": int(len(values)),
            })
    return pd.DataFrame(rows)


def _paper_save_cross_run_metric_figures(metrics_df, summary_df):
    """Save canonical cross-run metric figures."""
    metric_order = [
        "Accuracy", "Weighted precision", "Weighted recall", "Weighted F1-score",
        "Macro AUC", "Weighted AUC", "Balanced accuracy",
    ]
    plot_df = summary_df[summary_df["Metric"].isin(metric_order)].copy()
    if not plot_df.empty:
        plot_df["Metric"] = pd.Categorical(plot_df["Metric"], categories=metric_order, ordered=True)
        plot_df = plot_df.sort_values("Metric")
        fig, ax = plt.subplots(figsize=(9, 5.2))
        x = np.arange(len(plot_df))
        bars = ax.bar(x, plot_df["Mean"], yerr=plot_df["SD"], capsize=4,
                      color="#4C78A8", edgecolor="black", linewidth=0.7)
        ax.set_xticks(x)
        ax.set_xticklabels(plot_df["Metric"], rotation=25, ha="right")
        ax.set_ylim(0, 1.05)
        ax.set_ylabel("Score")
        ax.set_title(f"{experiment_display_name()}: test-set metrics (mean +/- SD across runs)")
        for bar, value in zip(bars, plot_df["Mean"]):
            ax.text(bar.get_x() + bar.get_width() / 2, min(value + 0.025, 1.03),
                    f"{value:.3f}", ha="center", va="bottom", fontsize=8)
        fig.tight_layout()
        fig.savefig(os.path.join(PAPER_ARTIFACT_DIR, "figure_01_test_metrics_mean_sd.png"),
                    bbox_inches="tight")
        plt.close(fig)

    metric_cols = [c for c in metric_order if c in metrics_df.columns and not metrics_df[c].isna().all()]
    if metric_cols:
        long_df = metrics_df.melt(id_vars=["Run", "Seed"], value_vars=metric_cols,
                                  var_name="Metric", value_name="Score").dropna()
        if not long_df.empty:
            fig, ax = plt.subplots(figsize=(9, 5.2))
            sns.boxplot(data=long_df, x="Metric", y="Score", ax=ax, color="#72B7B2")
            sns.stripplot(data=long_df, x="Metric", y="Score", ax=ax,
                          color="black", size=4, jitter=0.08)
            ax.set_ylim(0, 1.05)
            ax.set_xlabel("")
            ax.set_ylabel("Score")
            ax.set_title(f"{experiment_display_name()}: test-set metric variability across runs")
            ax.tick_params(axis="x", rotation=25)
            for tick in ax.get_xticklabels():
                tick.set_ha("right")
            fig.tight_layout()
            fig.savefig(os.path.join(PAPER_ARTIFACT_DIR, "figure_02_test_metrics_distribution.png"),
                        bbox_inches="tight")
            plt.close(fig)


def _paper_save_per_class_f1(per_class_summary):
    """Save canonical per-class F1-score figure."""
    if per_class_summary.empty:
        return
    f1_df = per_class_summary[per_class_summary["Metric"] == "F1-score"].copy()
    if f1_df.empty:
        return
    fig, ax = plt.subplots(figsize=(8, 5.2))
    x = np.arange(len(f1_df))
    bars = ax.bar(x, f1_df["Mean"], yerr=f1_df["SD"], capsize=4,
                  color="#59A14F", edgecolor="black", linewidth=0.7)
    ax.set_xticks(x)
    ax.set_xticklabels(["\\n".join(textwrap.wrap(c, 18)) for c in f1_df["Class"]])
    ax.set_ylim(0, 1.05)
    ax.set_xlabel("Class")
    ax.set_ylabel("F1-score")
    ax.set_title(f"{experiment_display_name()}: per-class test-set F1-score (mean +/- SD across runs)")
    for bar, value in zip(bars, f1_df["Mean"]):
        ax.text(bar.get_x() + bar.get_width() / 2, min(value + 0.025, 1.03),
                f"{value:.3f}", ha="center", va="bottom", fontsize=8)
    fig.tight_layout()
    fig.savefig(os.path.join(PAPER_ARTIFACT_DIR, "figure_03_per_class_f1_score.png"),
                bbox_inches="tight")
    plt.close(fig)


def _paper_save_pooled_confusion_and_roc(runs):
    """Save pooled confusion matrix and ROC figures across all runs."""
    class_names_local = runs[0].get("class_names") or list(runs[0].get("per_class_metrics", {}).keys())
    if not class_names_local:
        return
    n_classes = len(class_names_local)
    cm_total = np.zeros((n_classes, n_classes), dtype=float)
    y_true_parts = []
    prob_parts = []
    for run_data in runs:
        y_true_local = _paper_safe_array(run_data.get("y_true"))
        y_pred_local = _paper_safe_array(run_data.get("y_pred"))
        if y_true_local is not None and y_pred_local is not None:
            cm_total += confusion_matrix(y_true_local, y_pred_local, labels=range(n_classes)).astype(float)
            y_true_parts.append(y_true_local)
        probs = _paper_safe_array(run_data.get("pred_probs"))
        if probs is not None and probs.ndim == 2 and probs.shape[1] == n_classes:
            prob_parts.append(probs)
    if cm_total.sum() > 0:
        cm_norm = cm_total / np.maximum(cm_total.sum(axis=1, keepdims=True), 1)
        labels = display_class_names(class_names_local, width=16)
        fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.8))
        sns.heatmap(cm_total.astype(int), annot=True, fmt="d", cmap="Blues",
                    xticklabels=labels, yticklabels=labels, ax=axes[0], cbar=True)
        sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
                    xticklabels=labels, yticklabels=labels, ax=axes[1], cbar=True)
        axes[0].set_title("Pooled confusion matrix counts")
        axes[1].set_title("Row-normalized recall matrix")
        for axis in axes:
            format_confusion_matrix_axes(axis)
        fig.suptitle(f"{experiment_display_name()}: pooled test-set confusion matrix analysis",
                     fontsize=12, fontweight="bold")
        fig.tight_layout(rect=[0, 0, 1, 0.90])
        fig.savefig(os.path.join(PAPER_ARTIFACT_DIR, "figure_04_pooled_confusion_matrices.png"),
                    bbox_inches="tight")
        plt.close(fig)
        pd.DataFrame(cm_total.astype(int), index=plain_class_names(class_names_local),
                     columns=plain_class_names(class_names_local)).to_csv(
            os.path.join(PAPER_ARTIFACT_DIR, "table_05_pooled_confusion_matrix_counts.csv"))
        pd.DataFrame(cm_norm, index=plain_class_names(class_names_local),
                     columns=plain_class_names(class_names_local)).to_csv(
            os.path.join(PAPER_ARTIFACT_DIR, "table_06_pooled_confusion_matrix_normalized.csv"))

    if y_true_parts and prob_parts and len(y_true_parts) == len(prob_parts):
        all_y_true = np.concatenate(y_true_parts)
        all_probs = np.concatenate(prob_parts)
        y_bin = label_binarize(all_y_true, classes=range(n_classes))
        auc_rows = []
        fig, ax = plt.subplots(figsize=(7.4, 6.0))
        for ci, class_name in enumerate(class_names_local):
            try:
                fpr, tpr, _ = roc_curve(y_bin[:, ci], all_probs[:, ci])
                auc_value = auc(fpr, tpr)
            except ValueError:
                continue
            auc_rows.append({"Class": format_class_label(class_name), "AUC": float(auc_value)})
            ax.plot(fpr, tpr, lw=2, label=f"{format_class_label(class_name)} (AUC={auc_value:.3f})")
        if auc_rows:
            ax.plot([0, 1], [0, 1], "--", color="0.55", lw=1)
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1.01)
            ax.set_xlabel("False positive rate")
            ax.set_ylabel("True positive rate")
            ax.set_title(f"{experiment_display_name()}: pooled one-vs-rest ROC analysis")
            ax.legend(loc="lower right", fontsize=8)
            fig.tight_layout()
            fig.savefig(os.path.join(PAPER_ARTIFACT_DIR, "figure_05_pooled_roc_curves_ovr.png"),
                        bbox_inches="tight")
            plt.close(fig)
            pd.DataFrame(auc_rows).to_csv(os.path.join(PAPER_ARTIFACT_DIR, "table_07_pooled_auc_scores.csv"),
                                          index=False)


def _paper_save_training_convergence(runs):
    """Save canonical training convergence figure across all runs."""
    runs_with_history = [r for r in runs if isinstance(r.get("history"), dict) and "loss" in r.get("history", {})]
    if not runs_with_history:
        return
    palette = sns.color_palette("deep", n_colors=max(3, len(runs_with_history)))
    fig, axes = plt.subplots(2, 2, figsize=(12.5, 8.5))
    best_epochs = []
    best_val_accs = []
    run_labels = []
    for color, run_data in zip(palette, runs_with_history):
        hist = run_data["history"]
        label = _paper_run_label(run_data)
        run_labels.append(label)
        if "val_accuracy" in hist:
            axes[0, 0].plot(hist["val_accuracy"], color=color, lw=2, label=label)
        if "val_loss" in hist:
            axes[0, 1].plot(hist["val_loss"], color=color, lw=2, label=label)
            best_idx = int(np.argmin(hist["val_loss"]))
        else:
            best_idx = int(np.argmin(hist["loss"]))
        best_epochs.append(best_idx + 1)
        best_val_accs.append(float(hist.get("val_accuracy", hist.get("accuracy", [np.nan]))[best_idx]))
        if "accuracy" in hist and "val_accuracy" in hist:
            gap = np.asarray(hist["accuracy"]) - np.asarray(hist["val_accuracy"])
            axes[1, 1].plot(gap, color=color, lw=2, label=label)
    axes[0, 0].set_title("Validation accuracy trajectory")
    axes[0, 0].set_xlabel("Epoch")
    axes[0, 0].set_ylabel("Accuracy")
    axes[0, 1].set_title("Validation loss trajectory")
    axes[0, 1].set_xlabel("Epoch")
    axes[0, 1].set_ylabel("Loss")
    axes[1, 0].bar(run_labels, best_epochs, color=palette[:len(best_epochs)], edgecolor="black")
    axes[1, 0].set_title("Best epoch by validation loss")
    axes[1, 0].set_ylabel("Epoch")
    axes[1, 0].tick_params(axis="x", rotation=20)
    axes[1, 1].set_title("Generalization gap (training - validation accuracy)")
    axes[1, 1].set_xlabel("Epoch")
    axes[1, 1].set_ylabel("Accuracy gap")
    for axis in [axes[0, 0], axes[0, 1], axes[1, 1]]:
        axis.legend(fontsize=7)
    fig.suptitle(f"{experiment_display_name()}: training convergence analysis across runs",
                 fontsize=12, fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    fig.savefig(os.path.join(PAPER_ARTIFACT_DIR, "figure_06_training_convergence.png"),
                bbox_inches="tight")
    plt.close(fig)
    pd.DataFrame({
        "Run label": run_labels,
        "Best epoch": best_epochs,
        "Validation accuracy at best epoch": best_val_accs,
    }).to_csv(os.path.join(PAPER_ARTIFACT_DIR, "table_08_training_convergence.csv"), index=False)


def _paper_save_single_run_artifacts(runs):
    """Save canonical per-run reports and figures for all completed runs."""
    for run_data in runs:
        out_dir = _paper_run_dir(run_data)
        class_names_local = run_data.get("class_names") or list(run_data.get("per_class_metrics", {}).keys())
        y_true_local = _paper_safe_array(run_data.get("y_true"))
        y_pred_local = _paper_safe_array(run_data.get("y_pred"))
        probs_local = _paper_safe_array(run_data.get("pred_probs"))
        metrics_df = _paper_metrics_dataframe([run_data])
        metrics_df.to_csv(os.path.join(out_dir, "table_01_test_metrics.csv"), index=False)
        if class_names_local and y_true_local is not None and y_pred_local is not None:
            with open(os.path.join(out_dir, "report_classification_metrics.txt"), "w", encoding="utf-8") as f:
                f.write(classification_report(
                    y_true_local, y_pred_local,
                    target_names=plain_class_names(class_names_local), digits=4))
            cm = confusion_matrix(y_true_local, y_pred_local, labels=range(len(class_names_local))).astype(float)
            cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
            labels = display_class_names(class_names_local, width=16)
            fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.8))
            sns.heatmap(cm.astype(int), annot=True, fmt="d", cmap="Blues",
                        xticklabels=labels, yticklabels=labels, ax=axes[0], cbar=True)
            sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
                        xticklabels=labels, yticklabels=labels, ax=axes[1], cbar=True)
            axes[0].set_title("Confusion matrix counts")
            axes[1].set_title("Row-normalized recall matrix")
            for axis in axes:
                format_confusion_matrix_axes(axis)
            fig.suptitle(f"{experiment_display_name()}: test-set confusion matrix analysis ({_paper_run_label(run_data)})",
                         fontsize=12, fontweight="bold")
            fig.tight_layout(rect=[0, 0, 1, 0.90])
            fig.savefig(os.path.join(out_dir, "figure_01_confusion_matrices.png"), bbox_inches="tight")
            plt.close(fig)
        if class_names_local and y_true_local is not None and probs_local is not None and probs_local.ndim == 2:
            y_bin = label_binarize(y_true_local, classes=range(len(class_names_local)))
            fig, ax = plt.subplots(figsize=(7.4, 6.0))
            auc_rows = []
            for ci, class_name in enumerate(class_names_local):
                try:
                    fpr, tpr, _ = roc_curve(y_bin[:, ci], probs_local[:, ci])
                    auc_value = auc(fpr, tpr)
                except ValueError:
                    continue
                auc_rows.append({"Class": format_class_label(class_name), "AUC": float(auc_value)})
                ax.plot(fpr, tpr, lw=2, label=f"{format_class_label(class_name)} (AUC={auc_value:.3f})")
            if auc_rows:
                ax.plot([0, 1], [0, 1], "--", color="0.55", lw=1)
                ax.set_xlim(0, 1)
                ax.set_ylim(0, 1.01)
                ax.set_xlabel("False positive rate")
                ax.set_ylabel("True positive rate")
                ax.set_title(f"One-vs-rest ROC analysis ({_paper_run_label(run_data)})")
                ax.legend(loc="lower right", fontsize=8)
                fig.tight_layout()
                fig.savefig(os.path.join(out_dir, "figure_02_roc_curves_ovr.png"), bbox_inches="tight")
                plt.close(fig)
                pd.DataFrame(auc_rows).to_csv(os.path.join(out_dir, "table_02_auc_scores.csv"), index=False)
        hist = run_data.get("history")
        if isinstance(hist, dict) and "loss" in hist:
            fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
            axes[0].plot(hist.get("loss", []), label="Training loss", lw=2)
            if "val_loss" in hist:
                axes[0].plot(hist.get("val_loss", []), label="Validation loss", lw=2)
            axes[0].set_title("Loss trajectories")
            axes[0].set_xlabel("Epoch")
            axes[0].set_ylabel("Loss")
            axes[0].legend()
            axes[1].plot(hist.get("accuracy", []), label="Training accuracy", lw=2)
            if "val_accuracy" in hist:
                axes[1].plot(hist.get("val_accuracy", []), label="Validation accuracy", lw=2)
            axes[1].set_title("Accuracy trajectories")
            axes[1].set_xlabel("Epoch")
            axes[1].set_ylabel("Accuracy")
            axes[1].legend()
            fig.suptitle(f"{experiment_display_name()}: learning curves ({_paper_run_label(run_data)})",
                         fontsize=12, fontweight="bold")
            fig.tight_layout(rect=[0, 0, 1, 0.90])
            fig.savefig(os.path.join(out_dir, "figure_03_learning_curves.png"), bbox_inches="tight")
            plt.close(fig)
        report_lines = [
            "PAPER-READY SINGLE-RUN EVALUATION REPORT",
            "=" * 80,
            f"Model/strategy: {experiment_display_name()}",
            f"Run: {run_data.get('run')}",
            f"Seed: {run_data.get('seed')}",
            f"Source result directory: {run_data.get('result_dir')}",
            f"Canonical artifact directory: {out_dir}",
            "",
            "Generated files:",
            "- table_01_test_metrics.csv",
            "- report_classification_metrics.txt",
            "- figure_01_confusion_matrices.png",
            "- figure_02_roc_curves_ovr.png when probability outputs are available",
            "- figure_03_learning_curves.png when training history is available",
        ]
        with open(os.path.join(out_dir, "report_single_run_evaluation.txt"), "w", encoding="utf-8") as f:
            f.write("\\n".join(report_lines))


def export_paper_ready_artifacts(runs):
    """Export canonical tables, figures, and reports for paper writing."""
    if not runs:
        raise RuntimeError("all_runs_results is empty. Execute the training loop first.")
    _apply_paper_style()
    metrics_df = _paper_metrics_dataframe(runs)
    summary_df = _paper_metrics_summary(metrics_df)
    per_class_df = _paper_per_class_dataframe(runs)
    per_class_summary = _paper_per_class_summary(per_class_df)

    metrics_df.to_csv(os.path.join(PAPER_ARTIFACT_DIR, "table_01_test_metrics_by_run.csv"), index=False)
    summary_df.to_csv(os.path.join(PAPER_ARTIFACT_DIR, "table_02_test_metrics_summary.csv"), index=False)
    per_class_df.to_csv(os.path.join(PAPER_ARTIFACT_DIR, "table_03_per_class_metrics_by_run.csv"), index=False)
    per_class_summary.to_csv(os.path.join(PAPER_ARTIFACT_DIR, "table_04_per_class_metrics_summary.csv"), index=False)

    _paper_save_cross_run_metric_figures(metrics_df, summary_df)
    _paper_save_per_class_f1(per_class_summary)
    _paper_save_pooled_confusion_and_roc(runs)
    _paper_save_training_convergence(runs)
    _paper_save_single_run_artifacts(runs)

    report_lines = [
        "PAPER-READY MULTI-RUN EVALUATION REPORT",
        "=" * 80,
        f"Model/strategy: {experiment_display_name()}",
        f"Number of runs: {len(runs)}",
        f"Seeds: {[r.get('seed') for r in runs]}",
        f"Canonical artifact directory: {PAPER_ARTIFACT_DIR}",
        "",
        "Canonical tables:",
        "- table_01_test_metrics_by_run.csv",
        "- table_02_test_metrics_summary.csv",
        "- table_03_per_class_metrics_by_run.csv",
        "- table_04_per_class_metrics_summary.csv",
        "- table_05_pooled_confusion_matrix_counts.csv when predictions are available",
        "- table_06_pooled_confusion_matrix_normalized.csv when predictions are available",
        "- table_07_pooled_auc_scores.csv when probability outputs are available",
        "- table_08_training_convergence.csv when training history is available",
        "",
        "Canonical figures:",
        "- figure_01_test_metrics_mean_sd.png",
        "- figure_02_test_metrics_distribution.png",
        "- figure_03_per_class_f1_score.png",
        "- figure_04_pooled_confusion_matrices.png",
        "- figure_05_pooled_roc_curves_ovr.png when probability outputs are available",
        "- figure_06_training_convergence.png when training history is available",
    ]
    if not summary_df.empty:
        report_lines.extend(["", "Metric summary (mean +/- SD):"])
        for _, row in summary_df.iterrows():
            report_lines.append(f"- {row['Metric']}: {row['Mean']:.4f} +/- {row['SD']:.4f} (n={int(row['N'])})")
    with open(os.path.join(PAPER_ARTIFACT_DIR, "report_multi_run_evaluation.txt"), "w", encoding="utf-8") as f:
        f.write("\\n".join(report_lines))
    print(f"Paper-ready artifacts exported -> {PAPER_ARTIFACT_DIR}")


export_paper_ready_artifacts(all_runs_results)


In [ ]:
# ========== REFERENCE-COMPATIBLE REPORT EXPORTS ========== #
# This cell makes this notebook emit the same report/output contract as
# efficientnetb4-ft-b3to7-cbloss-db1-fn-reviewed.ipynb. It generates real
# artifacts whenever the required data are available and writes an explicit
# status placeholder only for optional interpretability artifacts that require
# a model-specific visualization cell.

import json
import zipfile
import shutil
from pathlib import Path

from sklearn.metrics import (
    balanced_accuracy_score, cohen_kappa_score, matthews_corrcoef,
    classification_report, confusion_matrix, roc_curve, auc,
)
from sklearn.preprocessing import label_binarize

REFERENCE_OPTIONAL_FIGURES = [
    "gradcam_visualization.png",
    "gradcam_pp.png",
    "gradcam_pp_misclassified.png",
    "se_attention_maps.png",
    "tsne_features.png",
]


def _ref_display_model_name():
    """Return the model/strategy name used in reference-compatible reports."""
    return experiment_display_name() if "experiment_display_name" in globals() else globals().get("STRATEGY_LABEL", "Garlic classification model")


def _ref_plain_labels(class_names_local):
    """Return publication-facing class labels."""
    return plain_class_names(class_names_local) if "plain_class_names" in globals() else [str(c).replace("_", " ") for c in class_names_local]


def _ref_display_labels(class_names_local, width=16):
    """Return wrapped class labels for compact figures."""
    return display_class_names(class_names_local, width=width) if "display_class_names" in globals() else _ref_plain_labels(class_names_local)


def _ref_metric(run_data, *keys):
    """Return the first available scalar metric from run_data."""
    for key in keys:
        value = run_data.get(key)
        if value is not None:
            try:
                return float(value)
            except (TypeError, ValueError):
                return value
    return np.nan


def _ref_get_class_names(run_data=None):
    """Infer class names from run data or stored per-class metrics."""
    if run_data and run_data.get("class_names"):
        return list(run_data["class_names"])
    if all_runs_results and all_runs_results[0].get("class_names"):
        return list(all_runs_results[0]["class_names"])
    if all_runs_results and all_runs_results[0].get("per_class_metrics"):
        return list(all_runs_results[0]["per_class_metrics"].keys())
    raise RuntimeError("Cannot infer class names from all_runs_results.")


def _ref_array(value):
    """Convert a stored value to a numpy array when it exists."""
    if value is None:
        return None
    return np.asarray(value)


def _ref_run_label(run_data):
    """Return a compact run label."""
    return f"Run {run_data.get('run', '?')} (seed {run_data.get('seed', '?')})"


def _ref_save_placeholder_png(path, title, message):
    """Save an explicit status figure for optional artifacts not generated here."""
    if os.path.exists(path):
        return False
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.axis("off")
    ax.text(0.5, 0.62, title, ha="center", va="center", fontsize=13, fontweight="bold")
    ax.text(0.5, 0.40, message, ha="center", va="center", fontsize=10, wrap=True)
    fig.tight_layout()
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return True


def _ref_copy_if_available(result_dir, candidates, target_name):
    """Create a standard alias from the first existing candidate file."""
    target = os.path.join(result_dir, target_name)
    if os.path.exists(target):
        return "exists"
    for name in candidates:
        src = os.path.join(result_dir, name)
        if os.path.exists(src):
            shutil.copy2(src, target)
            return f"copied from {name}"
    return "missing"


def _ref_run_metrics_rows(run_data):
    """Build run-level metric rows using the reference metric names."""
    return {
        "run": run_data.get("run"),
        "seed": run_data.get("seed"),
        "accuracy": _ref_metric(run_data, "accuracy"),
        "precision": _ref_metric(run_data, "precision"),
        "recall": _ref_metric(run_data, "recall"),
        "f1_score": _ref_metric(run_data, "f1_score"),
        "auc_ovr_macro": _ref_metric(run_data, "auc_ovr_macro", "auc_macro"),
        "auc_ovr_weighted": _ref_metric(run_data, "auc_ovr_weighted", "auc_weighted"),
        "bal_acc": _ref_metric(run_data, "bal_acc"),
        "kappa": _ref_metric(run_data, "kappa"),
        "mcc": _ref_metric(run_data, "mcc"),
    }


def _ref_export_run_level_files(run_data):
    """Export reference-compatible files inside one run directory."""
    result_dir = run_data.get("result_dir")
    if not result_dir:
        return []
    os.makedirs(result_dir, exist_ok=True)
    class_names_local = _ref_get_class_names(run_data)
    plain_labels = _ref_plain_labels(class_names_local)
    y_true_local = _ref_array(run_data.get("y_true"))
    y_pred_local = _ref_array(run_data.get("y_pred"))
    pred_probs_local = _ref_array(run_data.get("pred_probs"))
    history_local = run_data.get("history")
    manifest = []

    # Standard raw arrays used by downstream report notebooks.
    if y_true_local is not None:
        np.save(os.path.join(result_dir, "y_true.npy"), y_true_local)
        manifest.append(("y_true.npy", "generated"))
    if y_pred_local is not None:
        np.save(os.path.join(result_dir, "y_pred.npy"), y_pred_local)
        manifest.append(("y_pred.npy", "generated"))
    if pred_probs_local is not None:
        np.save(os.path.join(result_dir, "pred_probs.npy"), pred_probs_local)
        manifest.append(("pred_probs.npy", "generated"))
    if isinstance(history_local, dict):
        np.save(os.path.join(result_dir, "history.npy"), history_local, allow_pickle=True)
        manifest.append(("history.npy", "generated"))
    if run_data.get("test_filenames") is not None:
        with open(os.path.join(result_dir, "test_filenames.json"), "w", encoding="utf-8") as f:
            json.dump(list(run_data["test_filenames"]), f, ensure_ascii=False, indent=2)
        manifest.append(("test_filenames.json", "generated"))

    # Standard model alias.
    model_status = _ref_copy_if_available(
        result_dir,
        ["best_model.keras", "efficientnetb4_best.keras", "densenet121_best.keras", "inceptionv3_best.keras", "mobilenetv2_best.keras", "nasnetmobile_best.keras"],
        "best_model.keras",
    )
    manifest.append(("best_model.keras", model_status))

    if y_true_local is not None and y_pred_local is not None:
        report_text = classification_report(y_true_local, y_pred_local, target_names=plain_labels, digits=4)
        with open(os.path.join(result_dir, "classification_report.txt"), "w", encoding="utf-8") as f:
            f.write(report_text)
        with open(os.path.join(result_dir, "classification_report_readable.txt"), "w", encoding="utf-8") as f:
            f.write(report_text)
        cm = confusion_matrix(y_true_local, y_pred_local, labels=range(len(class_names_local)))
        fig, ax = plt.subplots(figsize=(7.2, 6.2))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=_ref_display_labels(class_names_local),
                    yticklabels=_ref_display_labels(class_names_local), ax=ax)
        if "format_confusion_matrix_axes" in globals():
            format_confusion_matrix_axes(ax)
        else:
            ax.set_xlabel("Predicted class"); ax.set_ylabel("True class")
        ax.set_title(f"{_ref_display_model_name()}: test-set confusion matrix ({_ref_run_label(run_data)})")
        fig.tight_layout()
        fig.savefig(os.path.join(result_dir, "confusion_matrix.png"), dpi=300, bbox_inches="tight")
        fig.savefig(os.path.join(result_dir, "confusion_matrix_standardized.png"), dpi=300, bbox_inches="tight")
        plt.close(fig)
        manifest.extend([
            ("classification_report.txt", "generated"),
            ("classification_report_readable.txt", "generated"),
            ("confusion_matrix.png", "generated"),
            ("confusion_matrix_standardized.png", "generated"),
        ])

        # Per-class metrics and top misclassification CSV.
        report_dict = classification_report(y_true_local, y_pred_local, target_names=class_names_local, output_dict=True, digits=4)
        per_rows = []
        for raw_name, label in zip(class_names_local, plain_labels):
            d = report_dict.get(raw_name, {})
            per_rows.append({
                "Class": label,
                "Source label": raw_name,
                "Precision": d.get("precision", np.nan),
                "Recall": d.get("recall", np.nan),
                "F1-Score": d.get("f1-score", np.nan),
                "Support": d.get("support", np.nan),
            })
        pd.DataFrame(per_rows).to_csv(os.path.join(result_dir, "per_class_metrics.csv"), index=False)
        pd.DataFrame(per_rows).to_csv(os.path.join(result_dir, "per_class_metrics_readable.csv"), index=False)
        wrong_rows = []
        wrong_idx = np.where(y_true_local != y_pred_local)[0]
        if pred_probs_local is not None and pred_probs_local.ndim == 2:
            confidences = pred_probs_local[wrong_idx, y_pred_local[wrong_idx]] if len(wrong_idx) else []
            ordered = wrong_idx[np.argsort(-confidences)] if len(wrong_idx) else []
        else:
            ordered = wrong_idx
        for rank, idx in enumerate(ordered[:50], start=1):
            row = {
                "rank": rank,
                "sample_index": int(idx),
                "true_class": plain_labels[int(y_true_local[idx])],
                "pred_class": plain_labels[int(y_pred_local[idx])],
                "true_source_label": class_names_local[int(y_true_local[idx])],
                "pred_source_label": class_names_local[int(y_pred_local[idx])],
            }
            if pred_probs_local is not None and pred_probs_local.ndim == 2:
                row["pred_confidence"] = float(pred_probs_local[idx, y_pred_local[idx]])
                row["true_class_probability"] = float(pred_probs_local[idx, y_true_local[idx]])
            if run_data.get("test_filenames") is not None:
                row["filename"] = list(run_data["test_filenames"])[idx]
            wrong_rows.append(row)
        pd.DataFrame(wrong_rows).to_csv(os.path.join(result_dir, "top5_misclassified_report.csv"), index=False)
        manifest.extend([
            ("per_class_metrics.csv", "generated"),
            ("per_class_metrics_readable.csv", "generated"),
            ("top5_misclassified_report.csv", "generated"),
        ])

    if y_true_local is not None and pred_probs_local is not None and pred_probs_local.ndim == 2:
        y_bin = label_binarize(y_true_local, classes=range(len(class_names_local)))
        fig, ax = plt.subplots(figsize=(7.5, 6.0))
        auc_rows = []
        for ci, label in enumerate(plain_labels):
            try:
                fpr, tpr, _ = roc_curve(y_bin[:, ci], pred_probs_local[:, ci])
                auc_value = auc(fpr, tpr)
            except ValueError:
                continue
            auc_rows.append({"Class": label, "AUC": float(auc_value)})
            ax.plot(fpr, tpr, lw=2, label=f"{label} (AUC={auc_value:.3f})")
        if auc_rows:
            ax.plot([0, 1], [0, 1], "--", color="0.55", lw=1)
            ax.set_xlim(0, 1); ax.set_ylim(0, 1.01)
            ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
            ax.set_title(f"One-vs-rest ROC analysis ({_ref_run_label(run_data)})")
            ax.legend(loc="lower right", fontsize=8)
            fig.tight_layout()
            fig.savefig(os.path.join(result_dir, "roc_curves_ovr.png"), dpi=300, bbox_inches="tight")
            fig.savefig(os.path.join(result_dir, "roc_curves_ovr_standardized.png"), dpi=300, bbox_inches="tight")
            plt.close(fig)
            pd.DataFrame(auc_rows).to_csv(os.path.join(result_dir, "auc_scores_readable.csv"), index=False)
            manifest.extend([
                ("roc_curves_ovr.png", "generated"),
                ("roc_curves_ovr_standardized.png", "generated"),
                ("auc_scores_readable.csv", "generated"),
            ])

    if isinstance(history_local, dict) and "loss" in history_local:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
        axes[0].plot(history_local.get("loss", []), label="Training loss", lw=2)
        if "val_loss" in history_local:
            axes[0].plot(history_local.get("val_loss", []), label="Validation loss", lw=2)
        axes[0].set_title("Loss trajectories")
        axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].legend(); axes[0].grid(alpha=0.3)
        axes[1].plot(history_local.get("accuracy", []), label="Training accuracy", lw=2)
        if "val_accuracy" in history_local:
            axes[1].plot(history_local.get("val_accuracy", []), label="Validation accuracy", lw=2)
        axes[1].set_title("Accuracy trajectories")
        axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy"); axes[1].legend(); axes[1].grid(alpha=0.3)
        fig.suptitle(f"{_ref_display_model_name()}: learning curves ({_ref_run_label(run_data)})", fontsize=12, fontweight="bold")
        fig.tight_layout(rect=[0, 0, 1, 0.90])
        fig.savefig(os.path.join(result_dir, "learning_curves.png"), dpi=300, bbox_inches="tight")
        fig.savefig(os.path.join(result_dir, "learning_curves_standardized.png"), dpi=300, bbox_inches="tight")
        plt.close(fig)
        manifest.extend([("learning_curves.png", "generated"), ("learning_curves_standardized.png", "generated")])
    else:
        _ref_copy_if_available(result_dir, ["learning_curve.png"], "learning_curves.png")

    # Optional interpretability artifacts: generate honest status placeholders only if missing.
    for fig_name in REFERENCE_OPTIONAL_FIGURES:
        status = "exists"
        if not os.path.exists(os.path.join(result_dir, fig_name)):
            # t-SNE has a common legacy name in several notebooks.
            if fig_name == "tsne_features.png" and os.path.exists(os.path.join(result_dir, "tsne_embedding.png")):
                shutil.copy2(os.path.join(result_dir, "tsne_embedding.png"), os.path.join(result_dir, fig_name))
                status = "copied from tsne_embedding.png"
            else:
                _ref_save_placeholder_png(
                    os.path.join(result_dir, fig_name),
                    fig_name.replace("_", " ").replace(".png", ""),
                    "This optional interpretability artifact is model-specific. Run the dedicated visualization cell to replace this status figure with the real analysis output.",
                )
                status = "status placeholder"
        manifest.append((fig_name, status))

    # Standard report text files.
    metrics_line = json.dumps(_ref_run_metrics_rows(run_data), ensure_ascii=False, indent=2)
    report_lines = [
        "REFERENCE-COMPATIBLE SINGLE-RUN REPORT",
        "=" * 80,
        f"Model/strategy: {_ref_display_model_name()}",
        f"Run: {run_data.get('run')}",
        f"Seed: {run_data.get('seed')}",
        f"Result directory: {result_dir}",
        "",
        "Metrics:",
        metrics_line,
    ]
    for name in ["RUN_EVALUATION_REPORT.txt", "SUMMARY_REPORT.txt"]:
        with open(os.path.join(result_dir, name), "w", encoding="utf-8") as f:
            f.write("\n".join(report_lines))
        manifest.append((name, "generated"))

    pd.DataFrame(manifest, columns=["artifact", "status"]).to_csv(
        os.path.join(result_dir, "reference_artifact_manifest.csv"), index=False)
    return manifest


def _ref_export_dataset_distribution():
    """Generate class distribution across dataset splits CSV/figure compatible with the reference notebook."""
    rows = []
    data_dir = globals().get("DATA_DIR")
    split_names = ["train", "val", "test"]
    if data_dir and os.path.isdir(data_dir):
        for split in split_names:
            split_dir = os.path.join(data_dir, split)
            if not os.path.isdir(split_dir):
                continue
            for class_name in sorted(os.listdir(split_dir)):
                class_dir = os.path.join(split_dir, class_name)
                if os.path.isdir(class_dir):
                    count = sum(
                        1 for fn in os.listdir(class_dir)
                        if fn.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".tiff"))
                    )
                    rows.append({"Split": split, "Class": _ref_plain_labels([class_name])[0], "Source label": class_name, "Count": count})
    if not rows and all_runs_results:
        first = all_runs_results[0]
        rows = [
            {"Split": "train", "Class": "All classes", "Source label": "all", "Count": first.get("n_train", np.nan)},
            {"Split": "val", "Class": "All classes", "Source label": "all", "Count": first.get("n_val", np.nan)},
            {"Split": "test", "Class": "All classes", "Source label": "all", "Count": first.get("n_test", np.nan)},
        ]
    dist_df = pd.DataFrame(rows)
    dist_df.to_csv(os.path.join(BASE_RESULT_DIR, "dataset_distribution.csv"), index=False)
    if not dist_df.empty:
        fig, ax = plt.subplots(figsize=(8.5, 5.2))
        sns.barplot(data=dist_df, x="Class", y="Count", hue="Split", ax=ax)
        ax.set_title(f"{_ref_display_model_name()}: class distribution across dataset splits")
        ax.set_xlabel("Class"); ax.set_ylabel("Number of images")
        ax.tick_params(axis="x", rotation=25)
        for tick in ax.get_xticklabels(): tick.set_ha("right")
        fig.tight_layout()
        fig.savefig(os.path.join(BASE_RESULT_DIR, "dataset_distribution.png"), dpi=300, bbox_inches="tight")
        plt.close(fig)


def export_reference_compatible_reports():
    """Export the full reference-compatible report contract."""
    if not all_runs_results:
        raise RuntimeError("all_runs_results is empty. Execute the training loop before this cell.")
    os.makedirs(BASE_RESULT_DIR, exist_ok=True)
    class_names_local = _ref_get_class_names(all_runs_results[0])
    plain_labels = _ref_plain_labels(class_names_local)

    # Per-run exports and base progress summaries.
    manifest_rows = []
    for run_data in all_runs_results:
        for artifact, status in _ref_export_run_level_files(run_data):
            manifest_rows.append({
                "run": run_data.get("run"),
                "seed": run_data.get("seed"),
                "artifact": artifact,
                "status": status,
                "result_dir": run_data.get("result_dir"),
            })
    pd.DataFrame(manifest_rows).to_csv(os.path.join(BASE_RESULT_DIR, "reference_artifact_manifest.csv"), index=False)

    progress_df = pd.DataFrame([_ref_run_metrics_rows(r) for r in all_runs_results])
    progress_df.to_csv(os.path.join(BASE_RESULT_DIR, "progress_summary.csv"), index=False)
    progress_df.to_csv(os.path.join(BASE_RESULT_DIR, "strategy_summary.csv"), index=False)
    progress_df.assign(strategy=globals().get("STRATEGY_KEY", _ref_display_model_name())).to_csv(
        os.path.join(BASE_RESULT_DIR, "summary.csv"), index=False)

    # Additional metrics summary.
    additional_rows = []
    for run_data in all_runs_results:
        y_true_local = _ref_array(run_data.get("y_true"))
        y_pred_local = _ref_array(run_data.get("y_pred"))
        row = _ref_run_metrics_rows(run_data)
        if y_true_local is not None and y_pred_local is not None:
            row["bal_acc"] = float(balanced_accuracy_score(y_true_local, y_pred_local))
            row["kappa"] = float(cohen_kappa_score(y_true_local, y_pred_local))
            row["mcc"] = float(matthews_corrcoef(y_true_local, y_pred_local))
        additional_rows.append(row)
    additional_df = pd.DataFrame(additional_rows)
    additional_df.to_csv(os.path.join(BASE_RESULT_DIR, "additional_metrics.csv"), index=False)
    additional_df.describe(include="all").to_csv(os.path.join(BASE_RESULT_DIR, "additional_metrics_summary.csv"))

    _ref_export_dataset_distribution()

    # Pooled confusion matrix and ROC curves across all runs.
    y_true_parts, y_pred_parts, prob_parts = [], [], []
    for run_data in all_runs_results:
        yt = _ref_array(run_data.get("y_true")); yp = _ref_array(run_data.get("y_pred")); pp = _ref_array(run_data.get("pred_probs"))
        if yt is not None and yp is not None:
            y_true_parts.append(yt); y_pred_parts.append(yp)
        if yt is not None and pp is not None and pp.ndim == 2:
            prob_parts.append(pp)
    if y_true_parts and y_pred_parts:
        all_y_true = np.concatenate(y_true_parts)
        all_y_pred = np.concatenate(y_pred_parts)
        cm = confusion_matrix(all_y_true, all_y_pred, labels=range(len(class_names_local)))
        cm_norm = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1)
        fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.8))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=_ref_display_labels(class_names_local), yticklabels=_ref_display_labels(class_names_local), ax=axes[0])
        sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, xticklabels=_ref_display_labels(class_names_local), yticklabels=_ref_display_labels(class_names_local), ax=axes[1])
        axes[0].set_title("Pooled confusion matrix counts"); axes[1].set_title("Row-normalized recall matrix")
        for ax in axes:
            if "format_confusion_matrix_axes" in globals(): format_confusion_matrix_axes(ax)
            else: ax.set_xlabel("Predicted class"); ax.set_ylabel("True class")
        fig.suptitle(f"{_ref_display_model_name()}: pooled test-set confusion matrix analysis", fontsize=12, fontweight="bold")
        fig.tight_layout(rect=[0, 0, 1, 0.90])
        fig.savefig(os.path.join(BASE_RESULT_DIR, "aggregate_confusion_matrix.png"), dpi=300, bbox_inches="tight")
        plt.close(fig)

        correct_mask = all_y_true == all_y_pred
        fig, axes = plt.subplots(1, 2, figsize=(13.0, 5.0))
        if prob_parts and len(prob_parts) == len(y_true_parts):
            all_probs = np.concatenate(prob_parts)
            conf = np.max(all_probs, axis=1)
            axes[0].hist(conf[correct_mask], bins=20, alpha=0.7, label="Correct", edgecolor="black")
            axes[0].hist(conf[~correct_mask], bins=20, alpha=0.7, label="Misclassified", edgecolor="black")
        else:
            axes[0].bar(["Correct", "Misclassified"], [int(correct_mask.sum()), int((~correct_mask).sum())])
        axes[0].set_title("Prediction outcome counts"); axes[0].set_ylabel("Count"); axes[0].legend()
        class_acc = []
        for ci, label in enumerate(plain_labels):
            mask = all_y_true == ci
            class_acc.append((label, float(np.mean(all_y_pred[mask] == ci)) if mask.any() else np.nan))
        acc_df = pd.DataFrame(class_acc, columns=["Class", "Accuracy"]).sort_values("Accuracy")
        sns.barplot(data=acc_df, x="Accuracy", y="Class", ax=axes[1], color="#4C78A8")
        axes[1].set_xlim(0, 1.05); axes[1].set_title("Class-wise test accuracy")
        fig.tight_layout()
        fig.savefig(os.path.join(BASE_RESULT_DIR, "error_analysis.png"), dpi=300, bbox_inches="tight")
        plt.close(fig)

    if y_true_parts and prob_parts and len(y_true_parts) == len(prob_parts):
        all_y_true = np.concatenate(y_true_parts)
        all_probs = np.concatenate(prob_parts)
        y_bin = label_binarize(all_y_true, classes=range(len(class_names_local)))
        fig, ax = plt.subplots(figsize=(7.5, 6.0))
        auc_rows = []
        for ci, label in enumerate(plain_labels):
            try:
                fpr, tpr, _ = roc_curve(y_bin[:, ci], all_probs[:, ci])
                auc_value = auc(fpr, tpr)
            except ValueError:
                continue
            auc_rows.append({"Class": label, "AUC": float(auc_value)})
            ax.plot(fpr, tpr, lw=2, label=f"{label} (AUC={auc_value:.3f})")
        if auc_rows:
            ax.plot([0, 1], [0, 1], "--", color="0.55", lw=1)
            ax.set_xlim(0, 1); ax.set_ylim(0, 1.01)
            ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
            ax.set_title(f"{_ref_display_model_name()}: pooled one-vs-rest ROC analysis")
            ax.legend(loc="lower right", fontsize=8)
            fig.tight_layout()
            fig.savefig(os.path.join(BASE_RESULT_DIR, "roc_curves.png"), dpi=300, bbox_inches="tight")
            plt.close(fig)
            pd.DataFrame(auc_rows).to_csv(os.path.join(BASE_RESULT_DIR, "auc_scores.csv"), index=False)

    # Training convergence across runs.
    hist_runs = [r for r in all_runs_results if isinstance(r.get("history"), dict) and "loss" in r.get("history", {})]
    if hist_runs:
        fig, axes = plt.subplots(2, 2, figsize=(12.5, 8.5))
        palette = sns.color_palette("deep", n_colors=max(3, len(hist_runs)))
        best_epochs = []
        labels = []
        for color, run_data in zip(palette, hist_runs):
            hist = run_data["history"]; label = _ref_run_label(run_data); labels.append(label)
            if "val_accuracy" in hist: axes[0, 0].plot(hist["val_accuracy"], color=color, label=label, lw=2)
            if "val_loss" in hist: axes[0, 1].plot(hist["val_loss"], color=color, label=label, lw=2)
            best_idx = int(np.argmin(hist.get("val_loss", hist.get("loss"))))
            best_epochs.append(best_idx + 1)
            if "accuracy" in hist and "val_accuracy" in hist:
                axes[1, 1].plot(np.asarray(hist["accuracy"]) - np.asarray(hist["val_accuracy"]), color=color, label=label, lw=2)
        axes[0, 0].set_title("Validation accuracy trajectory"); axes[0, 0].set_xlabel("Epoch"); axes[0, 0].set_ylabel("Accuracy")
        axes[0, 1].set_title("Validation loss trajectory"); axes[0, 1].set_xlabel("Epoch"); axes[0, 1].set_ylabel("Loss")
        axes[1, 0].bar(labels, best_epochs, color=palette[:len(best_epochs)], edgecolor="black")
        axes[1, 0].set_title("Best epoch by validation loss"); axes[1, 0].tick_params(axis="x", rotation=20)
        axes[1, 1].set_title("Generalization gap (training - validation accuracy)"); axes[1, 1].set_xlabel("Epoch"); axes[1, 1].set_ylabel("Accuracy gap")
        for ax in [axes[0,0], axes[0,1], axes[1,1]]: ax.legend(fontsize=7); ax.grid(alpha=0.3)
        fig.suptitle(f"{_ref_display_model_name()}: training convergence analysis across runs", fontsize=12, fontweight="bold")
        fig.tight_layout(rect=[0,0,1,0.94])
        fig.savefig(os.path.join(BASE_RESULT_DIR, "convergence_analysis.png"), dpi=300, bbox_inches="tight")
        plt.close(fig)

    # Multi-run report files using reference names.
    report_lines = [
        "REFERENCE-COMPATIBLE MULTI-RUN REPORT",
        "=" * 80,
        f"Model/strategy: {_ref_display_model_name()}",
        f"Number of runs: {len(all_runs_results)}",
        f"Seeds: {[r.get('seed') for r in all_runs_results]}",
        f"Base result directory: {BASE_RESULT_DIR}",
        "",
        "Generated reference-compatible artifacts are listed in reference_artifact_manifest.csv.",
    ]
    with open(os.path.join(BASE_RESULT_DIR, "MULTI_RUN_SUMMARY_REPORT.txt"), "w", encoding="utf-8") as f:
        f.write("\n".join(report_lines))

    # Base qualitative archive when a qualitative_analysis directory exists.
    selected_result_dir = all_runs_results[-1].get("result_dir")
    if selected_result_dir:
        qa_dir = os.path.join(selected_result_dir, "qualitative_analysis")
        os.makedirs(qa_dir, exist_ok=True)
        with open(os.path.join(qa_dir, "analysis_notes.txt"), "w", encoding="utf-8") as f:
            f.write("Reference-compatible qualitative analysis directory. Replace placeholder figures by running dedicated visualization cells when available.\n")
        zip_path = os.path.join(BASE_RESULT_DIR, "qualitative_analysis.zip")
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
            for root_dir, _, files in os.walk(qa_dir):
                for filename in files:
                    full = os.path.join(root_dir, filename)
                    zf.write(full, arcname=os.path.relpath(full, qa_dir))

    print(f"Reference-compatible report contract exported -> {BASE_RESULT_DIR}")


export_reference_compatible_reports()


In [4]:
# ========== AGGREGATE RESULTS FROM ALL RUNS ========== #
print("\n" + "="*80)
print("📊 AGGREGATING RESULTS FROM ALL RUNS")
print("="*80 + "\n")

# Extract overall metrics
accuracies = [r['accuracy'] for r in all_runs_results]
precisions = [r['precision'] for r in all_runs_results]
recalls = [r['recall'] for r in all_runs_results]
f1_scores = [r['f1_score'] for r in all_runs_results]

# Calculate mean and std
overall_stats = {
    'Accuracy': {
        'mean': np.mean(accuracies),
        'std': np.std(accuracies),
        'values': accuracies
    },
    'Precision': {
        'mean': np.mean(precisions),
        'std': np.std(precisions),
        'values': precisions
    },
    'Recall': {
        'mean': np.mean(recalls),
        'std': np.std(recalls),
        'values': recalls
    },
    'F1-Score': {
        'mean': np.mean(f1_scores),
        'std': np.std(f1_scores),
        'values': f1_scores
    }
}

# Print summary
print("OVERALL METRICS ACROSS ALL RUNS:")
print("-" * 80)
for metric_name, stats in overall_stats.items():
    print(f"{metric_name:12s}: {stats['mean']:.4f} ± {stats['std']:.4f}")
    print(f"              Individual runs: {[f'{v:.4f}' for v in stats['values']]}")
print("-" * 80)

# Get class names from first run
class_names = list(all_runs_results[0]['per_class_metrics'].keys())

# Calculate per-class statistics
per_class_stats = {}
for class_name in class_names:
    per_class_stats[class_name] = {}
    for metric in ['precision', 'recall', 'f1-score']:
        values = [r['per_class_metrics'][class_name][metric] for r in all_runs_results]
        per_class_stats[class_name][metric] = {
            'mean': np.mean(values),
            'std': np.std(values),
            'values': values
        }

print("\n✅ Statistics calculated successfully!")


📊 AGGREGATING RESULTS FROM ALL RUNS

OVERALL METRICS ACROSS ALL RUNS:
--------------------------------------------------------------------------------
Accuracy    : 0.7912 ± 0.0081
              Individual runs: ['0.7870', '0.8025', '0.7840']
Precision   : 0.8041 ± 0.0020
              Individual runs: ['0.8014', '0.8046', '0.8062']
Recall      : 0.7912 ± 0.0081
              Individual runs: ['0.7870', '0.8025', '0.7840']
F1-Score    : 0.7925 ± 0.0072
              Individual runs: ['0.7879', '0.8027', '0.7870']
--------------------------------------------------------------------------------

✅ Statistics calculated successfully!


In [5]:
# ========== CREATE SCIENTIFIC REPORT TABLE ========== #
print("\n" + "="*80)
print("📋 CREATING SCIENTIFIC REPORT TABLES")
print("="*80 + "\n")

# Create overall metrics table
overall_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Mean': [overall_stats['Accuracy']['mean'],
             overall_stats['Precision']['mean'],
             overall_stats['Recall']['mean'],
             overall_stats['F1-Score']['mean']],
    'Std': [overall_stats['Accuracy']['std'],
            overall_stats['Precision']['std'],
            overall_stats['Recall']['std'],
            overall_stats['F1-Score']['std']],
    'Run 1': [accuracies[0], precisions[0], recalls[0], f1_scores[0]],
    'Run 2': [accuracies[1], precisions[1], recalls[1], f1_scores[1]],
    'Run 3': [accuracies[2], precisions[2], recalls[2], f1_scores[2]]
})

# Format for scientific presentation
overall_df['Mean ± SD'] = overall_df.apply(
    lambda row: f"{row['Mean']:.4f} ± {row['Std']:.4f}", axis=1
)

print("\n📊 TEST-SET CLASSIFICATION PERFORMANCE (3 RUNS)")
print("="*80)
print(overall_df[['Metric', 'Mean ± SD', 'Run 1', 'Run 2', 'Run 3']].to_string(index=False))
print("="*80)

# Save to CSV
overall_df.to_csv(os.path.join(BASE_RESULT_DIR, "overall_metrics_summary.csv"), index=False)

# Create per-class metrics table
per_class_rows = []
for class_name in class_names:
    for metric in ['precision', 'recall', 'f1-score']:
        stats = per_class_stats[class_name][metric]
        per_class_rows.append({
            'Class': class_name,
            'Metric': metric.capitalize(),
            'Mean': stats['mean'],
            'Std': stats['std'],
            'Mean ± SD': f"{stats['mean']:.4f} ± {stats['std']:.4f}",
            'Run 1': stats['values'][0],
            'Run 2': stats['values'][1],
            'Run 3': stats['values'][2]
        })

per_class_df = pd.DataFrame(per_class_rows)

print("\n\n📊 PER-CLASS CLASSIFICATION PERFORMANCE (3 RUNS)")
print("="*80)
# Display grouped by class
for class_name in class_names:
    class_data = per_class_df[per_class_df['Class'] == class_name]
    print(f"\n{class_name}:")
    print(class_data[['Metric', 'Mean ± SD']].to_string(index=False))
print("="*80)

# Save to CSV
per_class_df.to_csv(os.path.join(BASE_RESULT_DIR, "per_class_metrics_summary.csv"), index=False)

print("\n✅ Summary tables saved to CSV files!")


📋 CREATING SCIENTIFIC REPORT TABLES


📊 TEST-SET CLASSIFICATION PERFORMANCE (3 RUNS)
   Metric      Mean ± SD    Run 1    Run 2    Run 3
 Accuracy 0.7912 ± 0.0081 0.787037 0.802469 0.783951
Precision 0.8041 ± 0.0020 0.801406 0.804601 0.806179
   Recall 0.7912 ± 0.0081 0.787037 0.802469 0.783951
 F1-Score 0.7925 ± 0.0072 0.787886 0.802741 0.786972


📊 PER-CLASS CLASSIFICATION PERFORMANCE (3 RUNS)

Fully_Peeled_Garlic:
   Metric      Mean ± SD
Precision 0.7629 ± 0.0184
   Recall 0.7619 ± 0.0269
 F1-score 0.7618 ± 0.0088

Partially_Peeled_Garlic:
   Metric      Mean ± SD
Precision 0.7001 ± 0.0577
   Recall 0.8905 ± 0.0281
 F1-score 0.7815 ± 0.0274

Spoiled_Garlic:
   Metric      Mean ± SD
Precision 0.8783 ± 0.0232
   Recall 0.7675 ± 0.0242
 F1-score 0.8186 ± 0.0099

✅ Summary tables saved to CSV files!


In [6]:
# ========== GENERATE LATEX TABLE FOR PAPER ========== #
print("\n" + "="*80)
print("📄 GENERATING LATEX TABLES FOR SCIENTIFIC PAPER")
print("="*80 + "\n")

# Overall metrics LaTeX table
latex_overall = r"""\begin{table}[h]
\centering
\caption{Test-set classification performance for NASNetMobile (mean ± SD over three independent runs)}
\label{tab:nasnetmobile_overall}
\begin{tabular}{lcccc}
\hline
\textbf{Metric} & \textbf{Mean ± SD} & \textbf{Run 1} & \textbf{Run 2} & \textbf{Run 3} \\
\hline
"""

for _, row in overall_df.iterrows():
    latex_overall += f"{row['Metric']} & {row['Mean ± SD']} & {row['Run 1']:.4f} & {row['Run 2']:.4f} & {row['Run 3']:.4f} \\\\\n"

latex_overall += r"""\hline
\end{tabular}
\end{table}
"""

print("LaTeX Table - Overall Metrics:")
print(latex_overall)

# Per-class metrics LaTeX table (compact version for paper)
latex_per_class = r"""\begin{table}[h]
\centering
\caption{Per-class classification performance for NASNetMobile (mean ± SD over three independent runs)}
\label{tab:nasnetmobile_per_class}
\begin{tabular}{lccc}
\hline
\textbf{Class} & \textbf{Precision} & \textbf{Recall} & \textbf{F1-Score} \\
\hline
"""

for class_name in class_names:
    prec = per_class_stats[class_name]['precision']
    rec = per_class_stats[class_name]['recall']
    f1 = per_class_stats[class_name]['f1-score']
    latex_per_class += f"{class_name} & {prec['mean']:.4f} ± {prec['std']:.4f} & {rec['mean']:.4f} ± {rec['std']:.4f} & {f1['mean']:.4f} ± {f1['std']:.4f} \\\\\n"

latex_per_class += r"""\hline
\end{tabular}
\end{table}
"""

print("\n" + "="*80)
print("LaTeX Table - Per-Class Metrics:")
print(latex_per_class)

# Save LaTeX tables to file
with open(os.path.join(BASE_RESULT_DIR, "latex_tables.tex"), "w") as f:
    f.write("% Overall Metrics Table\n")
    f.write(latex_overall)
    f.write("\n\n% Per-Class Metrics Table\n")
    f.write(latex_per_class)

print("\n✅ LaTeX tables saved to 'latex_tables.tex'")


📄 GENERATING LATEX TABLES FOR SCIENTIFIC PAPER

LaTeX Table - Overall Metrics:
\begin{table}[h]
\centering
\caption{Test-set classification performance for NASNetMobile (mean ± SD over three independent runs)}
\label{tab:nasnetmobile_overall}
\begin{tabular}{lcccc}
\hline
\textbf{Metric} & \textbf{Mean ± SD} & \textbf{Run 1} & \textbf{Run 2} & \textbf{Run 3} \\
\hline
Accuracy & 0.7912 ± 0.0081 & 0.7870 & 0.8025 & 0.7840 \\
Precision & 0.8041 ± 0.0020 & 0.8014 & 0.8046 & 0.8062 \\
Recall & 0.7912 ± 0.0081 & 0.7870 & 0.8025 & 0.7840 \\
F1-Score & 0.7925 ± 0.0072 & 0.7879 & 0.8027 & 0.7870 \\
\hline
\end{tabular}
\end{table}


LaTeX Table - Per-Class Metrics:
\begin{table}[h]
\centering
\caption{Per-class classification performance for NASNetMobile (mean ± SD over three independent runs)}
\label{tab:nasnetmobile_per_class}
\begin{tabular}{lccc}
\hline
\textbf{Class} & \textbf{Precision} & \textbf{Recall} & \textbf{F1-Score} \\
\hline
Fully_Peeled_Garlic & 0.7629 ± 0.0184 & 0.7619 ± 0.02

In [ ]:
# ========== VISUALIZATION OF RESULTS ACROSS RUNS ========== #
print("\n" + "="*80)
print("📈 CREATING VISUALIZATIONS")
print("="*80 + "\n")

# 1. Bar plot with error bars for overall metrics
fig, ax = plt.subplots(figsize=(10, 6))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
means = [overall_stats[m]['mean'] for m in metrics]
stds = [overall_stats[m]['std'] for m in metrics]

x_pos = np.arange(len(metrics))
bars = ax.bar(x_pos, means, yerr=stds, capsize=10, alpha=0.8, color='steelblue', edgecolor='black')

ax.set_xlabel('Metrics', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title(f'{experiment_display_name()}: test-set metrics (mean +/- SD across runs)', fontsize=14, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(metrics)
ax.set_ylim([0, 1.05])
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for i, (mean, std) in enumerate(zip(means, stds)):
    ax.text(i, mean + std + 0.02, f'{mean:.4f}\n±{std:.4f}', 
            ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig(os.path.join(BASE_RESULT_DIR, "overall_metrics_barplot.png"), dpi=300, bbox_inches='tight')
plt.show()

# 2. Box plot showing distribution across runs
fig, ax = plt.subplots(figsize=(10, 6))

data_for_box = [accuracies, precisions, recalls, f1_scores]
bp = ax.boxplot(data_for_box, labels=metrics, patch_artist=True, showmeans=True,
                meanprops=dict(marker='D', markerfacecolor='red', markersize=8))

for patch in bp['boxes']:
    patch.set_facecolor('lightblue')
    patch.set_alpha(0.7)

ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Run-to-run variability of test-set metrics', fontsize=14, fontweight='bold')
ax.set_ylim([0, 1.05])
ax.grid(axis='y', alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig(os.path.join(BASE_RESULT_DIR, "metrics_boxplot.png"), dpi=300, bbox_inches='tight')
plt.show()

# 3. Per-class F1-Score comparison
fig, ax = plt.subplots(figsize=(12, 6))

class_f1_means = [per_class_stats[c]['f1-score']['mean'] for c in class_names]
class_f1_stds = [per_class_stats[c]['f1-score']['std'] for c in class_names]

x_pos = np.arange(len(class_names))
bars = ax.bar(x_pos, class_f1_means, yerr=class_f1_stds, capsize=5, 
              alpha=0.8, color='coral', edgecolor='black')

ax.set_xlabel('Class', fontsize=12, fontweight='bold')
ax.set_ylabel('F1-Score', fontsize=12, fontweight='bold')
ax.set_title('Per-class test-set F1-score (mean +/- SD across runs)', fontsize=14, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(display_class_names(class_names, width=16), rotation=35, ha='right')
ax.set_ylim([0, 1.05])
ax.grid(axis='y', alpha=0.3)

# Add value labels
for i, (mean, std) in enumerate(zip(class_f1_means, class_f1_stds)):
    ax.text(i, mean + std + 0.02, f'{mean:.3f}', 
            ha='center', va='bottom', fontsize=8)

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig(os.path.join(BASE_RESULT_DIR, "per_class_f1score.png"), dpi=300, bbox_inches='tight')
plt.show()

print("✅ All visualizations created and saved!")

In [ ]:
# ========== GENERATE COMPREHENSIVE SUMMARY REPORT ========== #
print("\n" + "="*80)
print("📝 GENERATING COMPREHENSIVE SUMMARY REPORT")
print("="*80 + "\n")

report_lines = []
report_lines.append("="*100)
report_lines.append(f"{experiment_display_name()} - MULTI-RUN EXPERIMENT REPORT")
report_lines.append("="*100)
report_lines.append(f"\nGenerated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
report_lines.append(f"Random Seeds: {RANDOM_SEEDS}")
report_lines.append(f"Number of Runs: {len(RANDOM_SEEDS)}")

report_lines.append("\n" + "="*100)
report_lines.append("TEST-SET CLASSIFICATION PERFORMANCE (mean ? SD)")
report_lines.append("="*100)
report_lines.append(f"{'Metric':<20} {'Mean ± SD':<25} {'Run 1':<15} {'Run 2':<15} {'Run 3':<15}")
report_lines.append("-"*100)

for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:
    stats = overall_stats[metric]
    report_lines.append(f"{metric:<20} {stats['mean']:.4f} ± {stats['std']:.4f}      " + 
                       f"{stats['values'][0]:<15.4f} {stats['values'][1]:<15.4f} {stats['values'][2]:<15.4f}")

report_lines.append("="*100)

report_lines.append("\n" + "="*100)
report_lines.append("PER-CLASS CLASSIFICATION PERFORMANCE (mean ? SD)")
report_lines.append("="*100)

for class_name in class_names:
    report_lines.append(f"\nClass: {class_name}")
    report_lines.append("-"*100)
    report_lines.append(f"{'Metric':<20} {'Mean ± SD':<25} {'Run 1':<15} {'Run 2':<15} {'Run 3':<15}")
    report_lines.append("-"*100)
    
    for metric in ['precision', 'recall', 'f1-score']:
        stats = per_class_stats[class_name][metric]
        report_lines.append(f"{metric.capitalize():<20} {stats['mean']:.4f} ± {stats['std']:.4f}      " + 
                           f"{stats['values'][0]:<15.4f} {stats['values'][1]:<15.4f} {stats['values'][2]:<15.4f}")

report_lines.append("\n" + "="*100)
report_lines.append("INDIVIDUAL RUN DETAILS")
report_lines.append("="*100)

for run_result in all_runs_results:
    report_lines.append(f"\nRun {run_result['run']} (Seed: {run_result['seed']})")
    report_lines.append("-"*100)
    report_lines.append(f"  Accuracy:  {run_result['accuracy']:.4f}")
    report_lines.append(f"  Precision: {run_result['precision']:.4f}")
    report_lines.append(f"  Recall:    {run_result['recall']:.4f}")
    report_lines.append(f"  F1-Score:  {run_result['f1_score']:.4f}")
    report_lines.append(f"  Result Dir: {run_result['result_dir']}")

report_lines.append("\n" + "="*100)
report_lines.append("STATISTICAL SUMMARY")
report_lines.append("="*100)
report_lines.append(f"\nBest Run (by Accuracy):")
best_run_idx = np.argmax(accuracies)
report_lines.append(f"  Run {best_run_idx + 1} (Seed: {RANDOM_SEEDS[best_run_idx]})")
report_lines.append(f"  Accuracy: {accuracies[best_run_idx]:.4f}")

report_lines.append(f"\nWorst Run (by Accuracy):")
worst_run_idx = np.argmin(accuracies)
report_lines.append(f"  Run {worst_run_idx + 1} (Seed: {RANDOM_SEEDS[worst_run_idx]})")
report_lines.append(f"  Accuracy: {accuracies[worst_run_idx]:.4f}")

report_lines.append(f"\nVariability (Coefficient of Variation):")
for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:
    cv = (overall_stats[metric]['std'] / overall_stats[metric]['mean']) * 100
    report_lines.append(f"  {metric}: {cv:.2f}%")

report_lines.append("\n" + "="*100)
report_lines.append("GENERATED FILES")
report_lines.append("="*100)
report_lines.append("  ✓ overall_metrics_summary.csv - Overall metrics in CSV format")
report_lines.append("  ✓ per_class_metrics_summary.csv - Per-class metrics in CSV format")
report_lines.append("  ✓ latex_tables.tex - LaTeX tables for scientific paper")
report_lines.append("  ✓ overall_metrics_barplot.png - Bar plot with error bars")
report_lines.append("  ✓ metrics_boxplot.png - Box plot showing distribution")
report_lines.append("  ✓ per_class_f1score.png - Per-class F1-score comparison")
report_lines.append(f"  📁 run_1_seed_{RANDOM_SEEDS[0]}/ - Full results from Run 1")
report_lines.append(f"  📁 run_2_seed_{RANDOM_SEEDS[1]}/ - Full results from Run 2")
report_lines.append(f"  📁 run_3_seed_{RANDOM_SEEDS[2]}/ - Full results from Run 3")

report_lines.append("\n" + "="*100)
report_lines.append("END OF REPORT")
report_lines.append("="*100)

# Print report
report_text = "\n".join(report_lines)
print(report_text)

# Save report
with open(os.path.join(BASE_RESULT_DIR, "MULTI_RUN_SUMMARY_REPORT.txt"), "w", encoding="utf-8") as f:
    f.write(report_text)

print("\n✅ Comprehensive summary report saved to 'MULTI_RUN_SUMMARY_REPORT.txt'")

In [9]:
# ========== ZIP ALL RESULTS ========== #
import shutil

print("\n" + "="*80)
print("🗜️  CREATING COMPLETE ARCHIVE")
print("="*80 + "\n")

zip_output_path = "/kaggle/working/NASNetMobile_MultiRun_Complete"
print(f"Source: {BASE_RESULT_DIR}")
print(f"Output: {zip_output_path}.zip")

# Create zip file
shutil.make_archive(zip_output_path, 'zip', BASE_RESULT_DIR)

zip_size = os.path.getsize(f"{zip_output_path}.zip") / (1024*1024)
print(f"\n✅ Complete multi-run report archived successfully!")
print(f"📦 Archive size: {zip_size:.2f} MB")
print(f"📍 Location: {zip_output_path}.zip")
print("\n" + "="*80)
print("🎉 ALL DONE! Ready for scientific paper submission!")
print("="*80)


🗜️  CREATING COMPLETE ARCHIVE

Source: /kaggle/working/report_NASNetMobile_MultiRun
Output: /kaggle/working/NASNetMobile_MultiRun_Complete.zip

✅ Complete multi-run report archived successfully!
📦 Archive size: 52.26 MB
📍 Location: /kaggle/working/NASNetMobile_MultiRun_Complete.zip

🎉 ALL DONE! Ready for scientific paper submission!


In [ ]:
# ========== LEARNING CURVES ========== #
plt.figure(figsize=(12,5))

# Accuracy
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title('Accuracy trajectories', fontweight='bold')
plt.legend()
plt.grid()

# Loss
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss trajectories', fontweight='bold')
plt.legend()
plt.grid()

plt.savefig(os.path.join(RESULT_DIR, "learning_curve.png"), dpi=300)
plt.show()

In [11]:
# ========== LOAD BEST MODEL ========== #
model = load_model(os.path.join(RESULT_DIR, 'nasnetmobile_best.keras'))

# predict
pred_probs = model.predict(test_generator, verbose=1)
y_pred = np.argmax(pred_probs, axis=1)
y_true = test_generator.classes
class_names = list(test_generator.class_indices.keys())

11/11 ━━━━━━━━━━━━━━━━━━━━ 45s 2s/step


In [12]:
# ========== CLASSIFICATION REPORT ========== #
report = classification_report(y_true, y_pred, target_names=class_names, digits=4)
print(report)

with open(os.path.join(RESULT_DIR, "classification_report.txt"), "w") as f:
    f.write(report)

                         precision    recall  f1-score   support

    Fully_Peeled_Garlic     0.7835    0.7238    0.7525       105
Partially_Peeled_Garlic     0.6289    0.9104    0.7439        67
         Spoiled_Garlic     0.9000    0.7697    0.8298       152

               accuracy                         0.7840       324
              macro avg     0.7708    0.8013    0.7754       324
           weighted avg     0.8062    0.7840    0.7870       324



In [ ]:
# ========== CONFUSION MATRIX ========== #
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(7,6))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=display_class_names(class_names),
            yticklabels=display_class_names(class_names),
            cmap='Blues')

plt.xlabel("Predicted")
plt.ylabel("True")
plt.title(f"{experiment_display_name()}: test-set confusion matrix", fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig(os.path.join(RESULT_DIR, "confusion_matrix.png"), dpi=300)
plt.show()

In [14]:
# ========== LOAD BEST MODEL ========== #
model = load_model(os.path.join(RESULT_DIR, 'nasnetmobile_best.keras'))

# reset generator (quan trọng)
test_generator.reset()

# predict
pred_probs = model.predict(test_generator, verbose=1)

y_pred = np.argmax(pred_probs, axis=1)
y_true = test_generator.classes
class_names = list(test_generator.class_indices.keys())

print("Total test samples:", len(y_true))

11/11 ━━━━━━━━━━━━━━━━━━━━ 50s 2s/step
Total test samples: 324


In [15]:
test_acc = np.mean(y_pred == y_true)
print("Test Accuracy:", test_acc)

Test Accuracy: 0.7839506172839507


In [16]:
# ========== PREPARE PATHS ========== #
test_dir = os.path.join(DATA_DIR, "test")
filepaths = [os.path.join(test_dir, f) for f in test_generator.filenames]

In [17]:
# ========== FIND CONFUSION TYPES ========== #
from collections import defaultdict

confusion_dict = defaultdict(list)

for i in range(len(y_true)):
    if y_true[i] != y_pred[i]:
        key = (class_names[y_true[i]], class_names[y_pred[i]])
        confidence = pred_probs[i][y_pred[i]]
        confusion_dict[key].append((filepaths[i], confidence, i))

print("Total confusion types:", len(confusion_dict))

Total confusion types: 6


In [18]:
# ========== SELECT REPRESENTATIVE IMAGES ========== #
import shutil

analysis_dir = os.path.join(RESULT_DIR, "qualitative_analysis")
os.makedirs(analysis_dir, exist_ok=True)

summary_lines = []

for (true_label, pred_label), samples in confusion_dict.items():

    # sort theo độ tự tin giảm dần
    samples_sorted = sorted(samples, key=lambda x: x[1], reverse=True)

    selected = samples_sorted[:10]  # lấy 2 ảnh

    pair_folder = os.path.join(analysis_dir, f"{true_label}_as_{pred_label}")
    os.makedirs(pair_folder, exist_ok=True)

    summary_lines.append(f"\n=== {true_label} → {pred_label} ===")

    for idx, (img_path, conf, i) in enumerate(selected):
        new_name = f"sample_{idx+1}_conf_{conf:.3f}.jpg"
        dst = os.path.join(pair_folder, new_name)
        shutil.copy(img_path, dst)

        summary_lines.append(f"{new_name} | confidence={conf:.3f}")

In [19]:
with open(os.path.join(analysis_dir, "analysis_notes.txt"), "w") as f:
    f.write("\n".join(summary_lines))

print("Saved qualitative analysis samples")

Saved qualitative analysis samples


In [20]:
!cd /kaggle/working/report_NASNetMobile_2809/qualitative_analysis && zip -r /kaggle/working/qualitative_analysis.zip .

/bin/bash: line 1: cd: /kaggle/working/report_NASNetMobile_2809/qualitative_analysis: No such file or directory


# ========== MODEL ANALYSIS & REPORTS ========== #

In [21]:
# ========== MODEL SUMMARY & PARAMETERS ========== #
model.summary()

# Count parameters
total_params = model.count_params()
trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
non_trainable_params = total_params - trainable_params

print("\n" + "="*60)
print(f"Total params: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")
print(f"Non-trainable params: {non_trainable_params:,}")
print("="*60)

# Save to file
with open(os.path.join(RESULT_DIR, "model_summary.txt"), "w", encoding="utf-8") as f:
    model.summary(print_fn=lambda x: f.write(x + '\n'))
    f.write("\n" + "="*60 + "\n")
    f.write(f"Total params: {total_params:,}\n")
    f.write(f"Trainable params: {trainable_params:,}\n")
    f.write(f"Non-trainable params: {non_trainable_params:,}\n")
    f.write("="*60 + "\n")

print("✅ Model summary saved")

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv1 (Conv2D) │ (None, 111, 111,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn1            │ (None, 111, 111,  │        128 │ stem_conv1[0][0]  │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 111, 111,  │          0 │ stem_bn1[0][0]    │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reduction_conv_1_s… │ (None, 111, 111,  │        352 │ activation[0][0]  │
│ (Conv2D)            │ 11)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reduction_bn_1_ste… │ (None, 111, 111,  │         44 │ reduction_conv_1… │
│ (BatchNormalizatio… │ 11)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 111, 111,  │          0 │ reduction_bn_1_s… │
│ (Activation)        │ 11)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 111, 111,  │          0 │ stem_bn1[0][0]    │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv_1_p… │ (None, 115, 115,  │          0 │ activation_1[0][… │
│ (ZeroPadding2D)     │ 11)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv_1_p… │ (None, 117, 117,  │          0 │ activation_3[0][… │
│ (ZeroPadding2D)     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv_1_r… │ (None, 56, 56,    │        396 │ separable_conv_1… │
│ (SeparableConv2D)   │ 11)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv_1_r… │ (None, 56, 56,    │      1,920 │ separable_conv_1… │
│ (SeparableConv2D)   │ 11)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv_1_b… │ (None, 56, 56,    │         44 │ separable_conv_1… │
│ (BatchNormalizatio… │ 11)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv_1_b… │ (None, 56, 56,    │         44 │ separable_conv_1… │
│ (BatchNormalizatio… │ 11)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 56, 56,    │          0 │ separable_conv_1… │
│ (Activation)        │ 11)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_4        │ (None, 56, 56,    │          0 │ separable_conv_1… │
│ (Activation)        │ 11)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv_2_r… │ (None, 56, 56,    │        396 │ activation_2[0][

 Total params: 4,685,214 (17.87 MB)

 Trainable params: 137,795 (538.26 KB)

 Non-trainable params: 4,271,828 (16.30 MB)

 Optimizer params: 275,591 (1.05 MB)


Total params: 4,409,623
Trainable params: 137,795
Non-trainable params: 4,271,828


✅ Model summary saved


In [22]:
# ========== MODEL SIZE ========== #
import tempfile

# Save model temporarily to get size
temp_model_path = os.path.join(tempfile.gettempdir(), "temp_model.keras")
model.save(temp_model_path)
model_size_bytes = os.path.getsize(temp_model_path)
model_size_mb = model_size_bytes / (1024 * 1024)

print("="*60)
print(f"Model size: {model_size_mb:.2f} MB ({model_size_bytes:,} bytes)")
print("="*60)

# Save to file
with open(os.path.join(RESULT_DIR, "model_size.txt"), "w", encoding="utf-8") as f:
    f.write("="*60 + "\n")
    f.write(f"Model size: {model_size_mb:.2f} MB ({model_size_bytes:,} bytes)\n")
    f.write("="*60 + "\n")

# Clean up
os.remove(temp_model_path)
print("✅ Model size saved")

Model size: 20.47 MB (21,460,158 bytes)
✅ Model size saved


In [23]:
# ========== INFERENCE SPEED ========== #
import time

# Warm-up: run a few predictions to initialize
print("Warming up...")
test_generator.reset()
warmup_batch = next(test_generator)[0][:5]  # 5 images
for _ in range(3):
    _ = model.predict(warmup_batch, verbose=0)

# Measure inference time on multiple batches
print("\nMeasuring inference speed...")
test_generator.reset()
num_test_batches = 10
total_images = 0
total_time = 0

for i in range(num_test_batches):
    batch_x, _ = next(test_generator)
    batch_size_actual = len(batch_x)
    
    start_time = time.time()
    _ = model.predict(batch_x, verbose=0)
    end_time = time.time()
    
    total_images += batch_size_actual
    total_time += (end_time - start_time)

# Calculate metrics
avg_time_per_image = (total_time / total_images) * 1000  # ms
fps = total_images / total_time

print("\n" + "="*60)
print("Inference Speed:")
print(f"  FPS: {fps:.2f}")
print(f"  ms/image: {avg_time_per_image:.2f}")
print(f"  Total images tested: {total_images}")
print(f"  Total time: {total_time:.3f}s")
print("="*60)

# Save to file
with open(os.path.join(RESULT_DIR, "inference_speed.txt"), "w", encoding="utf-8") as f:
    f.write("="*60 + "\n")
    f.write("Inference Speed:\n")
    f.write(f"  FPS: {fps:.2f}\n")
    f.write(f"  ms/image: {avg_time_per_image:.2f}\n")
    f.write(f"  Total images tested: {total_images}\n")
    f.write(f"  Total time: {total_time:.3f}s\n")
    f.write("="*60 + "\n")

print("✅ Inference speed saved")

Warming up...

Measuring inference speed...

Inference Speed:
  FPS: 13.42
  ms/image: 74.49
  Total images tested: 320
  Total time: 23.837s
✅ Inference speed saved


In [24]:
# ========== TOP-K ACCURACY ========== #
from sklearn.metrics import top_k_accuracy_score

top_1_acc = top_k_accuracy_score(y_true, pred_probs, k=1, labels=range(len(class_names)))
top_3_acc = top_k_accuracy_score(y_true, pred_probs, k=3, labels=range(len(class_names)))
top_5_acc = top_k_accuracy_score(y_true, pred_probs, k=5, labels=range(len(class_names)))

print("\n" + "="*60)
print("Top-K Accuracy:")
print(f"  Top-1 Accuracy: {top_1_acc:.4f} ({top_1_acc*100:.2f}%)")
print(f"  Top-3 Accuracy: {top_3_acc:.4f} ({top_3_acc*100:.2f}%)")
print(f"  Top-5 Accuracy: {top_5_acc:.4f} ({top_5_acc*100:.2f}%)")
print("="*60)

# Save to file
with open(os.path.join(RESULT_DIR, "topk_accuracy.txt"), "w", encoding="utf-8") as f:
    f.write("="*60 + "\n")
    f.write("Top-K Accuracy:\n")
    f.write(f"  Top-1 Accuracy: {top_1_acc:.4f} ({top_1_acc*100:.2f}%)\n")
    f.write(f"  Top-3 Accuracy: {top_3_acc:.4f} ({top_3_acc*100:.2f}%)\n")
    f.write(f"  Top-5 Accuracy: {top_5_acc:.4f} ({top_5_acc*100:.2f}%)\n")
    f.write("="*60 + "\n")

print("✅ Top-K accuracy saved")


Top-K Accuracy:
  Top-1 Accuracy: 0.7840 (78.40%)
  Top-3 Accuracy: 1.0000 (100.00%)
  Top-5 Accuracy: 1.0000 (100.00%)
✅ Top-K accuracy saved


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:2036: UndefinedMetricWarning: 'k' (3) greater than or equal to 'n_classes' (3) will result in a perfect score and is therefore meaningless.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:2036: UndefinedMetricWarning: 'k' (5) greater than or equal to 'n_classes' (3) will result in a perfect score and is therefore meaningless.
  warnings.warn(


In [25]:
# ========== PER-CLASS ACCURACY ========== #
from sklearn.metrics import classification_report

# Get per-class metrics
report_dict = classification_report(y_true, y_pred, target_names=class_names, 
                                   output_dict=True, digits=4)

# Create per-class accuracy dataframe
per_class_df = pd.DataFrame({
    'Class': class_names,
    'Precision': [report_dict[c]['precision'] for c in class_names],
    'Recall': [report_dict[c]['recall'] for c in class_names],
    'F1-Score': [report_dict[c]['f1-score'] for c in class_names],
    'Support': [report_dict[c]['support'] for c in class_names]
})

print("\n" + "="*60)
print("Per-Class Metrics:")
print(per_class_df.to_string(index=False))
print("="*60)

# Save to CSV
per_class_df.to_csv(os.path.join(RESULT_DIR, "per_class_metrics.csv"), index=False)
print("✅ Per-class metrics saved")


Per-Class Metrics:
                  Class  Precision   Recall  F1-Score  Support
    Fully_Peeled_Garlic   0.783505 0.723810  0.752475    105.0
Partially_Peeled_Garlic   0.628866 0.910448  0.743902     67.0
         Spoiled_Garlic   0.900000 0.769737  0.829787    152.0
✅ Per-class metrics saved


In [26]:
# ========== PREDICTIONS CSV ========== #
# Create detailed predictions dataframe
predictions_df = pd.DataFrame({
    'filename': test_generator.filenames,
    'true_label': [class_names[i] for i in y_true],
    'predicted_label': [class_names[i] for i in y_pred],
    'correct': y_true == y_pred,
    'confidence': [pred_probs[i][y_pred[i]] for i in range(len(y_pred))]
})

# Add top-3 predictions for each image
for k in range(min(3, len(class_names))):
    top_k_indices = np.argsort(pred_probs, axis=1)[:, -(k+1)]
    predictions_df[f'top_{k+1}_class'] = [class_names[i] for i in top_k_indices]
    predictions_df[f'top_{k+1}_prob'] = [pred_probs[i][top_k_indices[i]] for i in range(len(pred_probs))]

# Save to CSV
predictions_df.to_csv(os.path.join(RESULT_DIR, "predictions_detail.csv"), index=False)

print(f"✅ Predictions CSV saved ({len(predictions_df)} samples)")
print(f"   Correct predictions: {predictions_df['correct'].sum()}")
print(f"   Incorrect predictions: {(~predictions_df['correct']).sum()}")

✅ Predictions CSV saved (324 samples)
   Correct predictions: 254
   Incorrect predictions: 70


In [ ]:
# ========== COMPREHENSIVE SUMMARY REPORT ========== #
summary_report = []
summary_report.append("="*80)
summary_report.append(f"{experiment_display_name()} - COMPREHENSIVE EVALUATION REPORT")
summary_report.append("="*80)
summary_report.append(f"\nDataset: {DATA_DIR}")
summary_report.append(f"Result Directory: {RESULT_DIR}")
summary_report.append(f"Training Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")

summary_report.append("\n" + "-"*80)
summary_report.append("MODEL CONFIGURATION")
summary_report.append("-"*80)
summary_report.append(f"Architecture: NASNetMobile")
summary_report.append(f"Input Shape: {input_shape}")
summary_report.append(f"Number of Classes: {len(class_names)}")
summary_report.append(f"Classes: {', '.join(plain_class_names(class_names))}")
summary_report.append(f"\nTotal Parameters: {total_params:,}")
summary_report.append(f"Trainable Parameters: {trainable_params:,}")
summary_report.append(f"Non-trainable Parameters: {non_trainable_params:,}")
summary_report.append(f"Model Size: {model_size_mb:.2f} MB")

summary_report.append("\n" + "-"*80)
summary_report.append("DATASET STATISTICS")
summary_report.append("-"*80)
summary_report.append(f"Training Samples: {train_generator.samples}")
summary_report.append(f"Validation Samples: {val_generator.samples}")
summary_report.append(f"Test Samples: {test_generator.samples}")

summary_report.append("\n" + "-"*80)
summary_report.append("TRAINING CONFIGURATION")
summary_report.append("-"*80)
summary_report.append(f"Batch Size: {batch_size}")
summary_report.append(f"Total Epochs: {len(history.history['loss'])}")
summary_report.append(f"Initial Learning Rate: 1e-5")
summary_report.append(f"Optimizer: Adam with ExponentialDecay")
summary_report.append(f"Loss Function: CategoricalCrossentropy (label_smoothing=0.15)")
summary_report.append(f"Class Weights: Balanced")

summary_report.append("\n" + "-"*80)
summary_report.append("PERFORMANCE METRICS")
summary_report.append("-"*80)
summary_report.append(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
summary_report.append(f"Top-1 Accuracy: {top_1_acc:.4f} ({top_1_acc*100:.2f}%)")
summary_report.append(f"Top-3 Accuracy: {top_3_acc:.4f} ({top_3_acc*100:.2f}%)")
summary_report.append(f"Top-5 Accuracy: {top_5_acc:.4f} ({top_5_acc*100:.2f}%)")

summary_report.append("\n" + "-"*80)
summary_report.append("INFERENCE SPEED")
summary_report.append("-"*80)
summary_report.append(f"FPS: {fps:.2f}")
summary_report.append(f"ms/image: {avg_time_per_image:.2f}")

summary_report.append("\n" + "-"*80)
summary_report.append("BEST TRAINING EPOCH METRICS")
summary_report.append("-"*80)
best_val_loss_idx = np.argmin(history.history['val_loss'])
summary_report.append(f"Best Epoch: {best_val_loss_idx + 1}")
summary_report.append(f"  Train Loss: {history.history['loss'][best_val_loss_idx]:.4f}")
summary_report.append(f"  Train Accuracy: {history.history['accuracy'][best_val_loss_idx]:.4f}")
summary_report.append(f"  Val Loss: {history.history['val_loss'][best_val_loss_idx]:.4f}")
summary_report.append(f"  Val Accuracy: {history.history['val_accuracy'][best_val_loss_idx]:.4f}")

summary_report.append("\n" + "="*80)
summary_report.append("END OF REPORT")
summary_report.append("="*80)

# Print and save
summary_text = "\n".join(summary_report)
print(summary_text)

with open(os.path.join(RESULT_DIR, "SUMMARY_REPORT.txt"), "w", encoding="utf-8") as f:
    f.write(summary_text)

print("\n✅ Comprehensive summary report saved")

In [28]:
# ========== LIST ALL REPORT FILES ========== #
import glob

print("\n" + "="*80)
print("GENERATED REPORT FILES:")
print("="*80)

all_files = glob.glob(os.path.join(RESULT_DIR, "*"))
for file_path in sorted(all_files):
    if os.path.isfile(file_path):
        file_name = os.path.basename(file_path)
        file_size = os.path.getsize(file_path)
        if file_size < 1024:
            size_str = f"{file_size} B"
        elif file_size < 1024*1024:
            size_str = f"{file_size/1024:.2f} KB"
        else:
            size_str = f"{file_size/(1024*1024):.2f} MB"
        print(f"  ✓ {file_name:40s} ({size_str})")
    elif os.path.isdir(file_path):
        dir_name = os.path.basename(file_path)
        num_files = len([f for f in glob.glob(os.path.join(file_path, "**/*"), recursive=True) if os.path.isfile(f)])
        print(f"  📁 {dir_name:40s} ({num_files} files)")

print("="*80)


GENERATED REPORT FILES:
  ✓ SUMMARY_REPORT.txt                       (2.39 KB)
  ✓ classification_report.txt                (457 B)
  ✓ confusion_matrix.png                     (116.97 KB)
  ✓ inference_speed.txt                      (219 B)
  ✓ learning_curve.png                       (182.02 KB)
  ✓ model_size.txt                           (162 B)
  ✓ model_summary.txt                        (312.86 KB)
  ✓ nasnetmobile_best.keras                  (20.49 MB)
  ✓ per_class_metrics.csv                    (272 B)
  ✓ predictions_detail.csv                   (55.14 KB)
  📁 qualitative_analysis                     (47 files)
  ✓ topk_accuracy.txt                        (242 B)
  ✓ training_log.csv                         (3.92 KB)


In [29]:
# ========== ZIP ALL REPORTS ========== #
import shutil

zip_output_path = "/kaggle/working/NASNetMobile_Complete_Report"
print(f"\n🗜️  Creating complete report archive...")
print(f"Source: {RESULT_DIR}")
print(f"Output: {zip_output_path}.zip")

# Create zip file
shutil.make_archive(zip_output_path, 'zip', RESULT_DIR)

zip_size = os.path.getsize(f"{zip_output_path}.zip") / (1024*1024)
print(f"\n✅ Complete report archived successfully!")
print(f"📦 Archive size: {zip_size:.2f} MB")
print(f"📍 Location: {zip_output_path}.zip")
print("\n" + "="*80)
print("ARCHIVE CONTENTS:")
print("  ✓ SUMMARY_REPORT.txt - Comprehensive evaluation summary")
print("  ✓ model_summary.txt - Model architecture & parameters")
print("  ✓ model_size.txt - Model file size")
print("  ✓ inference_speed.txt - FPS & latency metrics")
print("  ✓ topk_accuracy.txt - Top-1, Top-3, Top-5 accuracy")
print("  ✓ classification_report.txt - Precision, Recall, F1 per class")
print("  ✓ per_class_metrics.csv - Detailed class metrics")
print("  ✓ predictions_detail.csv - All predictions with confidence")
print("  ✓ training_log.csv - Training history")
print("  ✓ learning_curve.png - Training visualization")
print("  ✓ confusion_matrix.png - Confusion matrix heatmap")
print("  ✓ nasnetmobile_best.keras - Best model weights")
print("  📁 qualitative_analysis/ - Misclassified samples analysis")
print("="*80)


🗜️  Creating complete report archive...
Source: /kaggle/working/report_NASNetMobile_MultiRun/run_3_seed_456
Output: /kaggle/working/NASNetMobile_Complete_Report.zip

✅ Complete report archived successfully!
📦 Archive size: 19.52 MB
📍 Location: /kaggle/working/NASNetMobile_Complete_Report.zip

ARCHIVE CONTENTS:
  ✓ SUMMARY_REPORT.txt - Comprehensive evaluation summary
  ✓ model_summary.txt - Model architecture & parameters
  ✓ model_size.txt - Model file size
  ✓ inference_speed.txt - FPS & latency metrics
  ✓ topk_accuracy.txt - Top-1, Top-3, Top-5 accuracy
  ✓ classification_report.txt - Precision, Recall, F1 per class
  ✓ per_class_metrics.csv - Detailed class metrics
  ✓ predictions_detail.csv - All predictions with confidence
  ✓ training_log.csv - Training history
  ✓ learning_curve.png - Training visualization
  ✓ confusion_matrix.png - Confusion matrix heatmap
  ✓ nasnetmobile_best.keras - Best model weights
  📁 qualitative_analysis/ - Misclassified samples analysis
